# Limpieza y unificación de recetas

## Contexto

**ChefWise** es un recomendador de comidas. El frontend (Angular) espera el contrato
de `frontend/src/app/models/recipe.model.ts`: nombre, categoría, minutos, porciones,
dificultad, ingredientes y pasos.

Este notebook deja las recetas listas para esa app: unifica las fuentes, limpia el texto
y escribe **un solo CSV** en `data/processed/`. El análisis exploratorio y la ingeniería de
variables están en `02_eda_e_ingenieria_de_variables.ipynb`; el modelo en `03_recomendador_content_based.ipynb`.

## Fuentes

1. **RecetasDeLaAbuela** — [somosnlp/RecetasDeLaAbuela](https://huggingface.co/datasets/somosnlp/RecetasDeLaAbuela).
2. **Kiwilimon** — scraping de [kiwilimon.com](https://www.kiwilimon.com/).

## Pipeline

`raw` → diagnóstico → limpieza inicial → unificación → ajustes de limpieza →
**limpieza de texto (marcas, autopromoción, pasos numerados)** → validación → **un solo archivo** en `processed`


In [1]:
import ast
import csv
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 140)


## 1. Rutas

El notebook funciona tanto si el kernel arranca en `chefWise/` como en `notebooks/`.


In [2]:
cwd = Path.cwd().resolve()
if (cwd / "data" / "raw").is_dir():
    PROJECT_DIR = cwd
elif (cwd.parent / "data" / "raw").is_dir():
    PROJECT_DIR = cwd.parent
else:
    raise FileNotFoundError(
        "No se encontró data/raw. Abre el notebook desde chefWise/ o notebooks/."
    )

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
ABUELA_PATH = RAW_DIR / "recetasdelaabuela.csv"
KIWI_PATH = RAW_DIR / "recetas_kiwilimon.csv"
OUTPUT_PATH = PROCESSED_DIR / "recetas_unificadas_limpias.csv"

print("Proyecto:", PROJECT_DIR)
print("RecetasDeLaAbuela:", ABUELA_PATH, "| existe:", ABUELA_PATH.exists())
print("Kiwilimon:", KIWI_PATH, "| existe:", KIWI_PATH.exists())
print("Salida:", OUTPUT_PATH)


Proyecto: /Users/i./Documents/DIPLOMADO/MODULO_5/chefWise
RecetasDeLaAbuela: /Users/i./Documents/DIPLOMADO/MODULO_5/chefWise/data/raw/recetasdelaabuela.csv | existe: True
Kiwilimon: /Users/i./Documents/DIPLOMADO/MODULO_5/chefWise/data/raw/recetas_kiwilimon.csv | existe: True
Salida: /Users/i./Documents/DIPLOMADO/MODULO_5/chefWise/data/processed/recetas_unificadas_limpias.csv


## 2. Diagnóstico: RecetasDeLaAbuela

Revisamos forma, nulos, columnas basura y campos que no significan lo mismo en todas las filas.


In [3]:
abuela_raw = pd.read_csv(ABUELA_PATH)

print("Shape:", abuela_raw.shape)
print("Columnas:", abuela_raw.columns.tolist())
print("Duplicados exactos:", int(abuela_raw.duplicated().sum()))
print("Id único:", abuela_raw["Id"].is_unique)

nulos = abuela_raw.isna().mean().mul(100).round(1).sort_values(ascending=False)
display(nulos.to_frame("% nulos"))
display(abuela_raw.head(2))


Shape: (20085, 19)
Columnas: ['Id', 'Nombre', 'URL', 'Ingredientes', 'Pasos', 'Pais', 'Duracion', 'Porciones', 'Calorias', 'Categoria', 'Contexto', 'Valoracion y Votos', 'Comensales', 'Tiempo', 'Dificultad', 'Categoria 2', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18']
Duplicados exactos: 0
Id único: True


,% nulos
Unnamed: 17,100.0
Unnamed: 16,100.0
Calorias,100.0
Categoria 2,55.2
Tiempo,47.4
Contexto,41.6
Comensales,37.4
Categoria,34.6
Duracion,34.0
Valoracion y Votos,33.2


,Id,Nombre,URL,Ingredientes,Pasos,Pais,Duracion,Porciones,Calorias,Categoria,Contexto,Valoracion y Votos,Comensales,Tiempo,Dificultad,Categoria 2,Unnamed: 16,Unnamed: 17,Unnamed: 18
0,1,Berenjenas rellenas,https://www.elmueble.com/cocinas/comidas-saludables_46627,"2 berenjenas, 1 pimiento rojo, 1 pimiento amarillo, 1 pimiento verde, 1 calabacín, 1 cebolleta, 100 ml de salsa de tomate, 4 huevos, 50 ...",Paso 1. Lava bien las berenjenas y pártelas por la mitad a lo largo. Vacíalas y cuécelas 5 minutos al vapor. Ponlas boca abajo para que ...,España,45 min,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Alto en fibra, Sin grasas trans, Sin sodio o sin sal",NaN,NaN,NaN
1,2,Alcachofas al horno con pico de gallo,https://www.elmueble.com/cocinas/comidas-saludables_46627,"4 alcachofas, 1 limón, 400 g de carne picada (mitad ternera, mitad cerdo), 1 cebolla blanca, 1 cebolla morada, 100 g de tomate frito, 2 ...","Paso 1. Precalentar el horno a 180?°C. , Paso 2. Pelar los tallos de las alcachofas, sin cortarlos; retirar las hojas exteriores y regar...",España,60 min,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Bajo en calorías, Sin grasa, Alto en fibra",NaN,NaN,NaN


### Hallazgos de esta fuente

- Hay tres columnas `Unnamed`. `Unnamed: 16` y `Unnamed: 17` están vacías.
  `Unnamed: 18` es un slug/título duplicado (exportación de RecetasGratis); no aporta señal nueva.
- `Calorias` está 100 % vacía → se descarta.
- `Porciones` **no es porciones** en la mayoría de filas: ~18k registros traen ahí
  una lista Python de ingredientes. Las porciones reales están en `Comensales`
  (`"4 comensales"`). Solo ~600 filas tienen un número en `Porciones`.
- `Tiempo` no es duración: es tipo de plato (`Plato principal`, `Desayuno`, `Entrante`…).
  La duración está en `Duracion`.
- `Ingredientes` y `Pasos` suelen ser listas serializadas (`['...']`); otras filas son
  texto separado por comas y etiquetas `Paso 1.`.
- Hay mojibake (`limÃ³n`) en algunos textos.
- `URL` cubre varios dominios; conviene conservar `source` por dominio.


In [4]:
print("Calorias no nulas:", int(abuela_raw["Calorias"].notna().sum()))
print("Unnamed:16/17 no nulas:", int(abuela_raw["Unnamed: 16"].notna().sum()), int(abuela_raw["Unnamed: 17"].notna().sum()))
print("Unnamed:18 no nulas:", int(abuela_raw["Unnamed: 18"].notna().sum()))

porciones = abuela_raw["Porciones"].dropna().astype(str)
print("\nPorciones que parecen lista:", int(porciones.str.startswith("[").sum()))
print("Porciones que parecen número:", int(porciones.str.fullmatch(r"\d+(?:\.\d+)?").sum()))

print("\nDominios:")
display(
    abuela_raw["URL"].astype(str).str.extract(r"https?://(?:www\.)?([^/]+)", expand=False)
    .value_counts(dropna=False)
    .head(10)
    .to_frame("recetas")
)
print("País:")
display(abuela_raw["Pais"].value_counts(dropna=False).head(12).to_frame("recetas"))
print("Dificultad:")
display(abuela_raw["Dificultad"].value_counts(dropna=False).to_frame("recetas"))
print("Tiempo (tipo de plato, no minutos):")
display(abuela_raw["Tiempo"].value_counts(dropna=False).head(12).to_frame("recetas"))
print("Ingredientes vacíos:", int(abuela_raw["Ingredientes"].isna().sum()))


Calorias no nulas: 0
Unnamed:16/17 no nulas: 0 0
Unnamed:18 no nulas: 18396

Porciones que parecen lista: 18187
Porciones que parecen número: 632

Dominios:


,recetas
URL,
recetasgratis.net,18396
yanuq.com,832
cookpad.com,720
directoalpaladar.com,92
elmueble.com,37
vitonica.com,8


País:


,recetas
Pais,
España,9486
Internacional,4436
Mexico,1922
Peru,1279
Perú,720
Argentina,632
NaN,496
Chile,251
Colombia,245


Dificultad:


,recetas
Dificultad,
Dificultad baja,10056
NaN,6303
Dificultad media,3010
Dificultad muy baja,399
Dificultad alta,305
Dificultad muy alta,12


Tiempo (tipo de plato, no minutos):


,recetas
Tiempo,
NaN,9527
Plato principal,3876
Entrante,2641
Postre,2061
Acompañamiento,674
Desayuno,530
Merienda,420
Cena,305
Cumpleaños,51


Ingredientes vacíos: 209


## 3. Diagnóstico: Kiwilimon

`pd.read_csv` falla: una fila del scrape pegó **dos recetas** (9 campos esperados, 17 encontrados).
Leemos con `csv.reader`, separamos esa fila y quitamos una cabecera repetida si aparece.


In [5]:
KIWI_COLUMNS = [
    "titulo", "dificultad", "tiempo", "tiempo_coccion",
    "ingredientes", "preparacion", "valoracion", "num_comentarios", "url",
]


def read_kiwilimon_with_repair(path: Path) -> tuple[pd.DataFrame, list]:
    rows = []
    malformed = []
    fused_pattern = re.compile(
        r"^(https?://www\.kiwilimon\.com/\S*?)"
        r"([A-ZÁÉÍÓÚÜÑ][^,]*?)\s*$"
    )

    expected_header = [c.lower() for c in KIWI_COLUMNS]
    with path.open(encoding="utf-8-sig", newline="", errors="replace") as f:
        for line_number, row in enumerate(csv.reader(f), start=1):
            if [c.strip().lower() for c in row] == expected_header:
                continue
            if len(row) == 9:
                rows.append(row)
                continue
            if len(row) == 17:
                malformed.append(line_number)
                fused = row[8]
                match = fused_pattern.match(fused)
                if not match:
                    raise ValueError(f"No se pudo reparar la fila {line_number}: {fused!r}")
                first_url, second_title = match.groups()
                first_row = row[:8] + [first_url]
                second_row = [second_title] + row[9:]
                if len(first_row) != 9 or len(second_row) != 9:
                    raise ValueError(f"Reparación inválida en fila {line_number}.")
                rows.extend([first_row, second_row])
                continue
            raise ValueError(f"Fila {line_number}: {len(row)} campos (se esperaban 9).")

    return pd.DataFrame(rows, columns=KIWI_COLUMNS), malformed


kiwi_raw, kiwi_repairs = read_kiwilimon_with_repair(KIWI_PATH)
header_like = kiwi_raw["titulo"].astype(str).str.strip().str.lower().eq("titulo")
kiwi_raw = kiwi_raw.loc[~header_like].reset_index(drop=True)

print("Shape tras reparación:", kiwi_raw.shape)
print("Líneas estructurales reparadas:", kiwi_repairs)
print("Cabeceras extra residuales eliminadas:", int(header_like.sum()))
print("Duplicados exactos:", int(kiwi_raw.duplicated().sum()))
print("Nulos por columna:")
display(kiwi_raw.replace("", np.nan).isna().sum().to_frame("nulos"))
display(kiwi_raw.head(3))
print("Dificultad:")
display(kiwi_raw["dificultad"].value_counts(dropna=False).to_frame("recetas"))


Shape tras reparación: (9020, 9)
Líneas estructurales reparadas: [2255]
Cabeceras extra residuales eliminadas: 0
Duplicados exactos: 0
Nulos por columna:


,nulos
titulo,0
dificultad,0
tiempo,0
tiempo_coccion,0
ingredientes,0
preparacion,2
valoracion,0
num_comentarios,0
url,0


,titulo,dificultad,tiempo,tiempo_coccion,ingredientes,preparacion,valoracion,num_comentarios,url
0,Brownies Pistache Nobles,Baja,20 mins,30 mins,"3/4 taza de harina, de avena, debe ser certificada libre de gluten (90 g) | 1/2 taza de cocoa, (42 g) | 3 cucharadas de fécula, de tapi...",Precalienta el horno a 170° C. | Engrasa el molde cuadrado de 24 cm y forra la base con papel. | En un tazón: cierne y mezcla los ingred...,5,1,https://www.kiwilimon.com/receta/postres/chocolate/brownies/brownies-pistache-nobles
1,Hojaldre con Queso de Cabra y Pinones,Baja,30 mins,0 mins,"150 gramos de masa de hojaldre, descongelada | 150 gramos de queso de cabra blanco | 1 taza de mermelada de chabacano | 1 cebolla, blanc...","En una olla derrita la mantequilla a fuego medio, agregue las cebollas y sazone con sal y pimienta. Cocine hasta que comienze a hervir l...",5,1,https://www.kiwilimon.com/receta/recetas-de-botanas-faciles/hojaldre-con-queso-de-cabra-y-pinones
2,Egg Cake con Espárragos,Baja,25 mins,20 mins,cantidad suficiente de aceite en aerosol | 12 claras de huevo | 1/2 taza de cebolla | 1/2 taza de pimiento morrón | 1/2 taza de espárra...,"Precalienta el horno a 175°C. | Rocía con un poco de aceite en aerosol el molde para cupcakes. | Agrega a una batidora las claras, la ce...",5,1,https://www.kiwilimon.com/receta/desayunos/egg-cake-con-esparragos


Dificultad:


,recetas
dificultad,
Baja,7637
Media,1271
Alta,112


## 4. Funciones de limpieza

Reglas:

- No reescribir recetas ni “corregir” ingredientes.
- Homogeneizar listas (`[...]` o `|`) a texto separado por ` | `.
- Reparar mojibake UTF-8 leído como Latin-1 cuando se pueda.
- Parsear tiempos a minutos y valoraciones a número.
- Mapear dificultad y tipo de comida al contrato de la app cuando sea razonable;
  si no hay evidencia, dejar nulo.


In [6]:
_MOJI_FIXED = (
    ("Ã¡", "á"), ("Ã©", "é"), ("Ãí", "í"), ("Ãó", "ó"), ("Ãú", "ú"),
    ("Ãñ", "ñ"), ("Ãü", "ü"), ("ÃÁ", "Á"), ("ÃÍ", "Í"),
    ("ÃÓ", "Ó"), ("ÃÚ", "Ú"), ("ÃÑ", "Ñ"), ("Ã ", "à"),
    ("Ã‰", "É"),  # Ã‰  (cp1252)
    ("Ãš", "Ú"),  # Ãš
)



def _repair_mojibake_pairs(text: str) -> str:
    """Corrige pares tipo Ã© mezclados con texto ya bien decodificado."""
    for src, dst in _MOJI_FIXED:
        if src in text:
            text = text.replace(src, dst)
    def repl(match):
        try:
            return match.group(0).encode("latin1").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            return match.group(0)
    text = re.sub("Ã[\xa0-\xff]", repl, text)
    text = re.sub("Â[\xa0-\xff ]", repl, text)
    return text


def _repair_mojibake_chunk(chunk: str) -> str:
    """Repara UTF-8 leído como Latin-1. Si falla el bloque completo, va por pares."""
    try:
        return chunk.encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return _repair_mojibake_pairs(chunk)


def fix_text_encoding(value) -> str:
    if pd.isna(value):
        return ""
    text = str(value).replace("\ufeff", "").strip()
    if any(marker in text for marker in ("Ã", "Â", "â", "ðŸ", "�")):
        text = _repair_mojibake_chunk(text)
        if "Ã" in text or "Â" in text:
            text = _repair_mojibake_pairs(text)
    return re.sub(r"\s+", " ", text).strip()


def parse_list_like(value):
    text = fix_text_encoding(value)
    if not text or text == "[]":
        return []
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [fix_text_encoding(x) for x in parsed if fix_text_encoding(x)]
        except (ValueError, SyntaxError):
            pass
    return None


def join_parts(parts) -> str:
    return " | ".join(p for p in (fix_text_encoding(x) for x in parts) if p)


def clean_ingredient_text(value) -> str:
    text = fix_text_encoding(value)
    parsed = parse_list_like(text)
    if parsed is not None:
        return join_parts(parsed)
    if "|" in text:
        return join_parts(text.split("|"))
    return text


def clean_instruction_text(value) -> str:
    text = fix_text_encoding(value)
    parsed = parse_list_like(text)
    if parsed is not None:
        cleaned = [re.sub(r"^\d+[\).:\-]\s*", "", p) for p in parsed]
        return join_parts(cleaned)
    if "|" in text:
        return join_parts(text.split("|"))
    if re.search(r"\bPaso\s*\d+", text, flags=re.I):
        parts = re.split(r"\bPaso\s*\d+\s*[.:\-]?\s*", text, flags=re.I)
        return join_parts(p.strip(" ,;") for p in parts)
    return text


def normalize_key(value) -> str:
    text = fix_text_encoding(value).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def parse_minutes(value):
    text = fix_text_encoding(value).lower()
    if not text:
        return np.nan
    text = text.replace("hrs", "h").replace("hr", "h")
    text = text.replace("mins", "m").replace("minutos", "m").replace("minuto", "m").replace("min", "m")
    text = text.replace(" ", "")
    hours = re.search(r"(\d+(?:\.\d+)?)h", text)
    minutes = re.search(r"(\d+(?:\.\d+)?)m", text)
    total = 0.0
    found = False
    if hours:
        total += float(hours.group(1)) * 60
        found = True
    if minutes:
        total += float(minutes.group(1))
        found = True
    if found:
        return total
    only = re.fullmatch(r"(\d+(?:\.\d+)?)", text)
    return float(only.group(1)) if only else np.nan


def parse_number(value):
    if pd.isna(value):
        return np.nan
    text = fix_text_encoding(value).lower().replace(",", ".")
    numbers = re.findall(r"\d+(?:\.\d+)?", text)
    if not numbers:
        return np.nan
    return float(np.mean([float(x) for x in numbers]))


def parse_servings(value):
    """Solo cantidades de ración, nunca listas de ingredientes coladas en Porciones."""
    if pd.isna(value):
        return np.nan
    text = fix_text_encoding(value)
    if not text or text.startswith("[") or "|" in text or text.count(",") > 2:
        return np.nan
    if re.search(r"[a-záéíóúñ]{4,}", text, flags=re.I) and "comensal" not in text.lower():
        if not re.search(r"\d", text):
            return np.nan
        if re.search(r"taza|cucharad|gramo|kilo|huevo|harina|aceite", text, flags=re.I):
            return np.nan
    return parse_number(text)


def parse_rating_and_votes(value):
    if pd.isna(value):
        return np.nan, np.nan
    text = fix_text_encoding(value).replace(",", ".")
    rating_match = re.search(r"Valoraci[oó]n\s*:\s*(\d+(?:\.\d+)?)", text, flags=re.I)
    votes_match = re.search(r"\((\d+)\s+voto", text, flags=re.I)
    rating = float(rating_match.group(1)) if rating_match else np.nan
    votes = int(votes_match.group(1)) if votes_match else np.nan
    return rating, votes


def parse_tags(value) -> str:
    text = fix_text_encoding(value)
    parsed = parse_list_like(text)
    if parsed is not None:
        return join_parts(parsed)
    if not text or text == "[]":
        return ""
    return join_parts(text.split(","))


def normalize_country(value) -> str:
    text = fix_text_encoding(value)
    key = normalize_key(text)
    mapping = {
        "mexico": "México",
        "peru": "Perú",
        "puerto rico": "Puerto Rico",
        "republica dominicana": "República Dominicana",
        "el salvador": "El Salvador",
        "costa rica": "Costa Rica",
        "espana": "España",
        "chile": "Chile",
        "colombia": "Colombia",
        "argentina": "Argentina",
        "venezuela": "Venezuela",
        "ecuador": "Ecuador",
        "guatemala": "Guatemala",
        "uruguay": "Uruguay",
        "bolivia": "Bolivia",
        "nicaragua": "Nicaragua",
        "cuba": "Cuba",
        "internacional": "Internacional",
    }
    return mapping.get(key, text) if text else ""


def source_from_url(url: str) -> str:
    match = re.search(r"https?://([^/]+)", fix_text_encoding(url).lower())
    domain = match.group(1).replace("www.", "") if match else ""
    return {
        "recetasgratis.net": "RecetasGratis",
        "yanuq.com": "Yanuq",
        "cookpad.com": "Cookpad",
        "directoalpaladar.com": "Directo al Paladar",
        "elmueble.com": "El Mueble",
        "vitonica.com": "Vitónica",
        "kiwilimon.com": "Kiwilimon",
    }.get(domain, domain or "Desconocido")


def root_kiwilimon_category(url: str) -> str:
    match = re.search(r"/receta/([^/]+)", fix_text_encoding(url).lower())
    if not match:
        return ""
    return match.group(1).replace("-", " ").strip()


KIWI_MEAL_MAP = {
    "desayunos": "desayuno",
    "postres": "postre",
    "postres para ninos": "postre",
    "bebidas": "bebida",
    "platos fuertes": "comida",
    "pollo y carne": "comida",
    "pescados y mariscos": "comida",
    "recetas de pastas": "comida",
    "ensaladas": "comida",
    "sopas": "comida",
    "guarniciones": "comida",
    "recetas de botanas faciles": "comida",
}

ABUELA_MEAL_MAP = {
    "desayuno": "desayuno",
    "merienda": "desayuno",
    "postre": "postre",
    "bebida": "bebida",
    "cóctel": "bebida",
    "coctel": "bebida",
    "plato principal": "comida",
    "entrante": "comida",
    "acompañamiento": "comida",
    "guarnicion": "comida",
    "guarnición": "comida",
    "sopa": "comida",
    "ensalada": "comida",
}


def map_app_category(meal_type: str, category_tags: str) -> str:
    key = normalize_key(meal_type)
    if key in ABUELA_MEAL_MAP:
        return ABUELA_MEAL_MAP[key]
    tags = normalize_key(category_tags)
    for slug, cat in KIWI_MEAL_MAP.items():
        if tags.startswith(normalize_key(slug)) or f" {normalize_key(slug)} " in f" {tags} ":
            return cat
    if "desayuno" in tags:
        return "desayuno"
    if "postre" in tags:
        return "postre"
    if "bebida" in tags or "coctel" in tags:
        return "bebida"
    if "cena" in key or "cena" in tags:
        return "cena"
    return ""


DIFFICULTY_APP = {
    "baja": "facil",
    "muy baja": "facil",
    "facil": "facil",
    "fácil": "facil",
    "media": "intermedio",
    "intermedio": "intermedio",
    "intermedia": "intermedio",
    "alta": "reto",
    "muy alta": "reto",
    "dificil": "reto",
    "difícil": "reto",
    "reto": "reto",
}


def map_app_difficulty(value) -> str:
    key = normalize_key(re.sub(r"^dificultad\s*", "", fix_text_encoding(value), flags=re.I))
    return DIFFICULTY_APP.get(key, "")


## 5. Limpieza de RecetasDeLaAbuela

- Se tiran columnas vacías o redundantes (`Calorias`, `Unnamed:*`).
- `Comensales` es la fuente de `servings`. `Porciones` solo se usa si es un número/rango,
  nunca si es una lista de ingredientes.
- `Categoria 2` se guarda como `diet_tags` (etiquetas nutricionales).
- `Contexto` se conserva para consulta, no entra al texto del recomendador base.


In [7]:
abuela = abuela_raw.copy()
drop_cols = [c for c in abuela.columns if str(c).startswith("Unnamed") or c == "Calorias"]
abuela = abuela.drop(columns=drop_cols)

abuela["name"] = abuela["Nombre"].map(fix_text_encoding)
abuela["url"] = abuela["URL"].map(fix_text_encoding)
abuela["ingredients"] = abuela["Ingredientes"].map(clean_ingredient_text)
abuela["instructions"] = abuela["Pasos"].map(clean_instruction_text)
abuela["source"] = abuela["url"].map(source_from_url)
abuela["country"] = abuela["Pais"].map(normalize_country)
abuela["meal_type"] = abuela["Tiempo"].map(fix_text_encoding)
abuela["difficulty"] = (
    abuela["Dificultad"].map(fix_text_encoding)
    .str.replace(r"^Dificultad\s*", "", regex=True)
    .str.strip()
)
abuela["total_time_min"] = abuela["Duracion"].map(parse_minutes)
abuela["servings"] = abuela["Comensales"].map(parse_servings)
abuela["servings"] = abuela["servings"].fillna(abuela["Porciones"].map(parse_servings))

ratings = abuela["Valoracion y Votos"].map(parse_rating_and_votes)
abuela["rating"] = ratings.map(lambda x: x[0])
abuela["rating_votes"] = ratings.map(lambda x: x[1])
abuela["category_tags"] = abuela["Categoria"].map(parse_tags)
abuela["diet_tags"] = abuela["Categoria 2"].map(parse_tags)
abuela["context"] = abuela["Contexto"].map(fix_text_encoding)
abuela["recipe_id_source"] = abuela["Id"].astype(str).str.strip()
abuela["prep_time_min"] = np.nan
abuela["cook_time_min"] = np.nan
abuela["num_comments"] = np.nan

abuela["name_key"] = abuela["name"].map(normalize_key)
abuela["ingredients_key"] = abuela["ingredients"].map(normalize_key)
abuela["has_ingredients"] = abuela["ingredients"].ne("")
abuela["has_instructions"] = abuela["instructions"].ne("")
abuela["app_category"] = [
    map_app_category(m, t) for m, t in zip(abuela["meal_type"], abuela["category_tags"])
]
abuela["app_difficulty"] = abuela["difficulty"].map(map_app_difficulty)

print("Filas:", len(abuela))
print("Sin ingredientes:", int((~abuela["has_ingredients"]).sum()))
print("Sin instrucciones:", int((~abuela["has_instructions"]).sum()))
print("Servings nulos:", int(abuela["servings"].isna().sum()))
print("\nFuentes:")
display(abuela["source"].value_counts(dropna=False).to_frame("recetas"))
print("app_category:")
display(abuela["app_category"].replace("", np.nan).value_counts(dropna=False).to_frame("recetas"))
print("app_difficulty:")
display(abuela["app_difficulty"].replace("", np.nan).value_counts(dropna=False).to_frame("recetas"))


Filas: 20085
Sin ingredientes: 210
Sin instrucciones: 211
Servings nulos: 6836

Fuentes:


,recetas
source,
RecetasGratis,18396
Yanuq,832
Cookpad,720
Directo al Paladar,92
El Mueble,37
Vitónica,8


app_category:


,recetas
app_category,
NaN,10252
comida,6517
postre,2061
desayuno,950
cena,305


app_difficulty:


,recetas
app_difficulty,
facil,10455
NaN,6303
intermedio,3010
reto,317


## 6. Limpieza de Kiwilimon

Ingredientes y pasos ya vienen con `|`. Se calcula `total_time_min` como prep + cocción
cuando hay al menos uno. El país se asigna como México (origen del sitio). La categoría
de la app se infiere del slug de la URL.


In [8]:
kiwi = kiwi_raw.copy()
kiwi["name"] = kiwi["titulo"].map(fix_text_encoding)
kiwi["url"] = kiwi["url"].map(fix_text_encoding)
kiwi["ingredients"] = kiwi["ingredientes"].map(clean_ingredient_text)
kiwi["instructions"] = kiwi["preparacion"].map(clean_instruction_text)
kiwi["source"] = "Kiwilimon"
kiwi["country"] = "México"
kiwi["difficulty"] = kiwi["dificultad"].map(fix_text_encoding)
kiwi["prep_time_min"] = kiwi["tiempo"].map(parse_minutes)
kiwi["cook_time_min"] = kiwi["tiempo_coccion"].map(parse_minutes)
kiwi["total_time_min"] = kiwi[["prep_time_min", "cook_time_min"]].sum(axis=1, min_count=1)
kiwi["rating"] = pd.to_numeric(kiwi["valoracion"], errors="coerce")
kiwi["rating_votes"] = np.nan
kiwi["num_comments"] = pd.to_numeric(kiwi["num_comentarios"], errors="coerce")
kiwi["category_tags"] = kiwi["url"].map(root_kiwilimon_category)
kiwi["diet_tags"] = ""
kiwi["meal_type"] = kiwi["category_tags"]
kiwi["servings"] = np.nan
kiwi["context"] = ""
kiwi["recipe_id_source"] = kiwi["url"].map(normalize_key)
kiwi["name_key"] = kiwi["name"].map(normalize_key)
kiwi["ingredients_key"] = kiwi["ingredients"].map(normalize_key)
kiwi["has_ingredients"] = kiwi["ingredients"].ne("")
kiwi["has_instructions"] = kiwi["instructions"].ne("")
kiwi["app_category"] = [
    map_app_category("", t) for t in kiwi["category_tags"]
]
kiwi["app_difficulty"] = kiwi["difficulty"].map(map_app_difficulty)

print("Filas:", len(kiwi))
print("Sin ingredientes:", int((~kiwi["has_ingredients"]).sum()))
print("Sin instrucciones:", int((~kiwi["has_instructions"]).sum()))
print("Dificultad:")
display(kiwi["difficulty"].value_counts(dropna=False).to_frame("recetas"))
print("app_category:")
display(kiwi["app_category"].replace("", np.nan).value_counts(dropna=False).to_frame("recetas"))
print("Top category_tags:")
display(kiwi["category_tags"].value_counts().head(12).to_frame("recetas"))


Filas: 9020
Sin ingredientes: 0
Sin instrucciones: 2
Dificultad:


,recetas
difficulty,
Baja,7637
Media,1271
Alta,112


app_category:


,recetas
app_category,
comida,4540
postre,2361
NaN,1054
bebida,651
desayuno,414


Top category_tags:


,recetas
category_tags,
postres,2315
pollo y carne,1142
recetas de botanas faciles,676
bebidas,650
pescados y mariscos,630
platos fuertes,577
guarniciones,482
desayunos,414
recetas de pastas,351


## 7. Esquema unificado

Pensado para tres usos:

| Bloque | Columnas |
| --- | --- |
| Contenido | `name`, `ingredients`, `instructions` |
| Filtros / app | `app_category`, `app_difficulty`, `total_time_min`, `servings`, `country`, `diet_tags` |
| Trazabilidad | `source`, `url`, `recipe_id_source`, `rating`, `rating_votes`, `num_comments` |

`app_category` y `app_difficulty` siguen el contrato de `recipe.model.ts`
(`desayuno|comida|cena|postre|bebida` y `facil|intermedio|reto`). Quedan vacíos
cuando la fuente no da evidencia suficiente (p. ej. casi no hay `cena` etiquetada).


In [9]:
COMMON_COLUMNS = [
    "recipe_id_source", "source", "name", "url",
    "ingredients", "instructions",
    "country", "meal_type", "app_category", "category_tags", "diet_tags",
    "difficulty", "app_difficulty",
    "prep_time_min", "cook_time_min", "total_time_min",
    "servings", "rating", "rating_votes", "num_comments", "context",
    "name_key", "ingredients_key", "has_ingredients", "has_instructions",
]

recipes = pd.concat(
    [abuela[COMMON_COLUMNS], kiwi[COMMON_COLUMNS]],
    ignore_index=True,
)

text_columns = [
    "source", "name", "url", "ingredients", "instructions", "country",
    "meal_type", "app_category", "category_tags", "diet_tags",
    "difficulty", "app_difficulty", "context", "name_key", "ingredients_key",
]
for col in text_columns:
    recipes[col] = recipes[col].map(fix_text_encoding)

nullable_text = [
    "country", "meal_type", "app_category", "category_tags", "diet_tags",
    "difficulty", "app_difficulty", "context", "instructions",
]
for col in nullable_text:
    recipes[col] = recipes[col].replace("", np.nan)

print("Combinado:", recipes.shape)
display(recipes.head(3))
print("Por fuente:")
display(recipes["source"].value_counts().to_frame("recetas"))


Combinado: (29105, 25)


,recipe_id_source,source,name,url,ingredients,instructions,country,meal_type,app_category,category_tags,diet_tags,difficulty,app_difficulty,prep_time_min,cook_time_min,total_time_min,servings,rating,rating_votes,num_comments,context,name_key,ingredients_key,has_ingredients,has_instructions
0,1,El Mueble,Berenjenas rellenas,https://www.elmueble.com/cocinas/comidas-saludables_46627,"2 berenjenas, 1 pimiento rojo, 1 pimiento amarillo, 1 pimiento verde, 1 calabacín, 1 cebolleta, 100 ml de salsa de tomate, 4 huevos, 50 ...",Lava bien las berenjenas y pártelas por la mitad a lo largo. Vacíalas y cuécelas 5 minutos al vapor. Ponlas boca abajo para que suelten ...,España,NaN,NaN,NaN,Alto en fibra | Sin grasas trans | Sin sodio o sin sal,NaN,NaN,NaN,NaN,45.0,4.0,NaN,NaN,NaN,NaN,berenjenas rellenas,2 berenjenas 1 pimiento rojo 1 pimiento amarillo 1 pimiento verde 1 calabacin 1 cebolleta 100 ml de salsa de tomate 4 huevos 50 gr de qu...,True,True
1,2,El Mueble,Alcachofas al horno con pico de gallo,https://www.elmueble.com/cocinas/comidas-saludables_46627,"4 alcachofas, 1 limón, 400 g de carne picada (mitad ternera, mitad cerdo), 1 cebolla blanca, 1 cebolla morada, 100 g de tomate frito, 2 ...","Precalentar el horno a 180?°C. | Pelar los tallos de las alcachofas, sin cortarlos; retirar las hojas exteriores y regarlas con el zumo ...",España,NaN,NaN,NaN,Bajo en calorías | Sin grasa | Alto en fibra,NaN,NaN,NaN,NaN,60.0,4.0,NaN,NaN,NaN,NaN,alcachofas al horno con pico de gallo,4 alcachofas 1 limon 400 g de carne picada mitad ternera mitad cerdo 1 cebolla blanca 1 cebolla morada 100 g de tomate frito 2 tomates m...,True,True
2,3,El Mueble,Arroz basmati salteado con Heura y verduras variadas al curry,https://www.elmueble.com/cocinas/comidas-saludables_46627,"200 g de arroz basmati integral, 450 ml de agua, 1 zanahoria, 1/2 pimiento rojo, 1 pimiento verde, 200 g de guisantes, 1/2 puerro, 250 g...","Hervir el arroz basmati durante unos 25 minutos, y una vez cocinado, colar y pasar por abundante agua fría, escurrir y reservar. | Corta...",España,NaN,NaN,NaN,Bajo en calorías | Alto en grasas | Bueno fuente de fibra,NaN,NaN,NaN,NaN,40.0,4.0,NaN,NaN,NaN,NaN,arroz basmati salteado con heura y verduras variadas al curry,200 g de arroz basmati integral 450 ml de agua 1 zanahoria 1 2 pimiento rojo 1 pimiento verde 200 g de guisantes 1 2 puerro 250 g de boc...,True,True


Por fuente:


,recetas
source,
RecetasGratis,18396
Kiwilimon,9020
Yanuq,832
Cookpad,720
Directo al Paladar,92
El Mueble,37
Vitónica,8


## 8. Registros no utilizables

Se elimina una fila si no tiene **nombre** o no tiene **ingredientes**.

Los ingredientes son la señal principal para un recomendador por contenido
(y para “con lo que tengo”). Una receta sin pasos sí puede quedarse: todavía
se puede recomendar y los pasos se pueden completar después.


In [10]:
before = len(recipes)
missing_name = recipes["name"].eq("")
missing_ingredients = recipes["ingredients"].eq("")
print("Sin nombre:", int(missing_name.sum()))
print("Sin ingredientes (con nombre):", int((~missing_name & missing_ingredients).sum()))
recipes = recipes.loc[~missing_name & ~missing_ingredients].copy()
print("Antes:", before, "| Después:", len(recipes), "| Eliminadas:", before - len(recipes))


Sin nombre: 0
Sin ingredientes (con nombre): 210
Antes: 29105 | Después: 28895 | Eliminadas: 210


## 9. Deduplicación conservadora

Mismo nombre en la misma fuente no implica duplicado (varias “arroz con pollo”).

Criterio: **misma fuente + nombre normalizado + ingredientes normalizados**.

No se usa solo la URL: páginas de colección a veces se asocian a varias recetas.


In [11]:
dedup_key = ["source", "name_key", "ingredients_key"]
n_dup = int(recipes.duplicated(dedup_key).sum())
recipes = recipes.drop_duplicates(dedup_key, keep="first").copy()
print("Duplicados eliminados:", n_dup)
print("Registros:", len(recipes))


Duplicados eliminados: 41
Registros: 28854


## 10. Variables para el recomendador

`recipe_text` junta nombre, ingredientes, categorías, dieta, tipo de comida, país y dificultad.

Las instrucciones **no** entran a `recipe_text` por defecto: el lenguaje operativo
(“mezcla”, “hornea”) hace que platos distintos se parezcan en TF-IDF.


In [12]:
def combine_for_recommender(row) -> str:
    parts = [
        row["name"],
        row["ingredients"],
        row["category_tags"] if pd.notna(row["category_tags"]) else "",
        row["diet_tags"] if pd.notna(row["diet_tags"]) else "",
        row["meal_type"] if pd.notna(row["meal_type"]) else "",
        row["app_category"] if pd.notna(row["app_category"]) else "",
        row["country"] if pd.notna(row["country"]) else "",
        row["difficulty"] if pd.notna(row["difficulty"]) else "",
    ]
    return " | ".join(fix_text_encoding(x) for x in parts if fix_text_encoding(x))


def count_structured_items(value):
    text = fix_text_encoding(value)
    if not text:
        return np.nan
    parsed = parse_list_like(text)
    if parsed is not None:
        return len(parsed)
    if "|" in text:
        return len([p for p in text.split("|") if p.strip()])
    return np.nan


recipes["recipe_text"] = recipes.apply(combine_for_recommender, axis=1)
recipes["n_ingredients_approx"] = recipes["ingredients"].map(count_structured_items)
recipes["n_steps_approx"] = recipes["instructions"].map(count_structured_items)
recipes.insert(0, "recipe_id", np.arange(1, len(recipes) + 1))

print("Columnas finales:", recipes.columns.tolist())


Columnas finales: ['recipe_id', 'recipe_id_source', 'source', 'name', 'url', 'ingredients', 'instructions', 'country', 'meal_type', 'app_category', 'category_tags', 'diet_tags', 'difficulty', 'app_difficulty', 'prep_time_min', 'cook_time_min', 'total_time_min', 'servings', 'rating', 'rating_votes', 'num_comments', 'context', 'name_key', 'ingredients_key', 'has_ingredients', 'has_instructions', 'recipe_text', 'n_ingredients_approx', 'n_steps_approx']


## 11. Validación


In [13]:
assert recipes["recipe_id"].is_unique
assert recipes["name"].ne("").all()
assert recipes["ingredients"].ne("").all()
assert not recipes.duplicated(dedup_key).any()
assert recipes["total_time_min"].dropna().ge(0).all()

bad_ratings = recipes["rating"].dropna().loc[lambda s: (s < 0) | (s > 5)]
print("Valoraciones fuera de 0–5:", len(bad_ratings))
assert bad_ratings.empty

valid_cat = {"desayuno", "comida", "cena", "postre", "bebida"}
valid_diff = {"facil", "intermedio", "reto"}
cats = set(recipes["app_category"].dropna().unique())
diffs = set(recipes["app_difficulty"].dropna().unique())
assert cats.issubset(valid_cat), cats
assert diffs.issubset(valid_diff), diffs

print("Por fuente:")
display(recipes["source"].value_counts().to_frame("recetas"))
print("Nulos relevantes:")
relevant = [
    "country", "meal_type", "app_category", "category_tags", "diet_tags",
    "difficulty", "app_difficulty", "prep_time_min", "cook_time_min",
    "total_time_min", "servings", "rating", "instructions",
]
display(recipes[relevant].isna().sum().sort_values(ascending=False).to_frame("nulos"))
print("Resumen numérico:")
display(
    recipes[
        ["total_time_min", "servings", "rating", "n_ingredients_approx", "n_steps_approx"]
    ].describe().T
)
print("Muestra unificada:")
display(recipes[["recipe_id", "source", "name", "app_category", "app_difficulty", "total_time_min", "servings"]].sample(5, random_state=42))


Valoraciones fuera de 0–5: 0
Por fuente:


,recetas
source,
RecetasGratis,18186
Kiwilimon,9019
Yanuq,792
Cookpad,720
Directo al Paladar,92
El Mueble,37
Vitónica,8


Nulos relevantes:


,nulos
diet_tags,20027
prep_time_min,19939
cook_time_min,19835
servings,15605
category_tags,12323
app_category,11056
meal_type,9277
total_time_min,6583
rating,6515
difficulty,6053


Resumen numérico:


,count,mean,std,min,25%,50%,75%,max
total_time_min,22271.0,70.249517,274.691662,0.0,25.0,40.0,60.0,30240.0
servings,13249.0,4.413692,3.206451,1.0,2.0,4.0,6.0,100.0
rating,22339.0,4.303723,0.763446,1.0,3.8,4.5,5.0,5.0
n_ingredients_approx,27868.0,9.027343,4.101308,2.0,6.0,8.0,11.0,50.0
n_steps_approx,27258.0,5.991782,2.815167,2.0,4.0,6.0,7.0,54.0


Muestra unificada:


,recipe_id,source,name,app_category,app_difficulty,total_time_min,servings
1527,1476,RecetasGratis,empanada en freidora de aire,comida,facil,15.0,2.0
11221,11035,RecetasGratis,pizza prosciutto casera,cena,facil,30.0,2.0
27798,27548,Kiwilimon,Bolitas de Nuez con Chocolate,postre,facil,10.0,NaN
24482,24233,Kiwilimon,Ensalada estilo oriental,comida,facil,20.0,NaN
28674,28424,Kiwilimon,Quesadillas de Masa,comida,facil,17.0,NaN


## 12. Ajustes de limpieza a partir del EDA

Hallazgos del EDA (ver `02_eda_e_ingenieria_de_variables.ipynb`) que afectan al recomendador:

1. **~38% sin `app_category`.** Muchas sí son desayunos, postres o comidas; solo faltaba
   inferirlas de `meal_type`, tags y el nombre. `Acompañamiento` y `panes` no se habían mapeado.
2. **Tiempos imposibles para una ficha.** `0 min` aparece cuando Kiwilimon pone `0 mins` de
   cocción y no hay prep. Cientos de `30h` / `60h` en cupcakes o tacos son casi seguro minutos
   scrapeados como horas. Tiempos de fermentación reales (vinagre, 24 h) no deben usarse como
   “minutos para cocinar hoy”.
3. **Encoding.** El arreglo Latin-1 fallaba si el mismo texto traía emoji; quedaban `azÃºcar`.
4. **Mismo platillo en dos fuentes** (mismo nombre e ingredientes normalizados) duplicaría
   resultados. Nos quedamos con la fila más completa.
5. **Nombres en minúsculas** (RecetasGratis) frente a título (Kiwilimon): se unifica la
   capitalización para la UI.

No se borran recetas solo por ser postre, bebida o fermentación: el catálogo sigue siendo
amplio. Lo que se corrige es la **señal que ve el modelo y la ficha**.


In [14]:
recipes_before = recipes.copy()
n_before = len(recipes)

# --- 1) Categoría de la app más completa ---
MEAL_MAP_EXTRA = {
    "acompanamiento": "comida",
    "acompañamiento": "comida",
    "guarnicion": "comida",
    "guarnición": "comida",
    "cumpleanos": "postre",
    "cumpleaños": "postre",
    "panes": "desayuno",
    "salsas": "comida",
    "comida para ninos": "comida",
    "recetas navidenas": "comida",
    "recetas a la parrilla": "comida",
    "recetas faciles": "",
    "saludables": "",
}

NAME_RULES = [
    (r"\b(smoothie|licuado|agua de|atole|cafe|café|mojito|coctel|cóctel|spritz|horchata|tepache|kombucha|ponche|latte|margarita)\b", "bebida"),
    (r"\b(cupcake|muffin|brownie|galleta|galletas|pastel|pay|flan|helado|mousse|cheesecake|churro|churros|dona|donas|bizcocho|budin|budín)\b", "postre"),
    (r"\b(hotcake|pancake|waffle|omelette|omelet|huevos revueltos|chilaquiles|desayuno|avena|granola|tostada francesa)\b", "desayuno"),
    (r"\b(taco|enchilada|sopa|caldo|ensalada|pollo|arroz|pasta|pizza|empanada|quesadilla|guiso|lenteja|frijol|carne|pescado|camaron|camarón|lasagna|lasaña|hamburguesa|sandwich|sándwich|ceviche|pozole|mole|burrito|fajita|crema de)\b", "comida"),
]


def infer_app_category(row) -> str:
    current = row["app_category"] if pd.notna(row.get("app_category")) else ""
    if current:
        return current
    meal = map_app_category(row.get("meal_type") or "", row.get("category_tags") or "")
    if meal:
        return meal
    extra = MEAL_MAP_EXTRA.get(normalize_key(row.get("meal_type") or ""), "")
    if extra:
        return extra
    extra = MEAL_MAP_EXTRA.get(normalize_key(row.get("category_tags") or ""), "")
    if extra:
        return extra
    name = fix_text_encoding(row.get("name") or "").lower()
    for pattern, cat in NAME_RULES:
        if re.search(pattern, name, flags=re.I):
            return cat
    tags = normalize_key(row.get("category_tags") or "")
    if "postre" in tags:
        return "postre"
    if "desayuno" in tags:
        return "desayuno"
    if "bebida" in tags:
        return "bebida"
    return ""


# --- 2) Tiempo usable para la ficha ---
def clean_listed_time(row):
    listed = row["total_time_min"]
    prep = row["prep_time_min"]
    cook = row["cook_time_min"]
    source = row["source"]

    if pd.notna(cook) and cook == 0:
        cook = np.nan
    if pd.notna(prep) and prep == 0:
        prep = np.nan

    # Kiwilimon: "30h" / "60h" con cocción 0 casi siempre son minutos mal etiquetados.
    if source == "Kiwilimon" and pd.notna(prep) and prep >= 25 * 60:
        hours = prep / 60.0
        if hours > 96:
            prep = np.nan  # fermentación de días: no es tiempo de ficha
        else:
            prep = hours  # 30h → 30 min

    parts = [x for x in (prep, cook) if pd.notna(x)]
    if source == "Kiwilimon":
        total = float(np.nansum(parts)) if parts else np.nan
        if total == 0:
            total = np.nan
    else:
        total = listed
        if pd.notna(total) and total == 0:
            total = np.nan

    usable = total
    if pd.notna(usable) and usable > 12 * 60:
        usable = np.nan  # reposo/fermentación: no se ofrece como "minutos de hoy"
    return pd.Series({"prep_time_min": prep, "cook_time_min": cook,
                      "listed_time_min": listed, "total_time_min": usable})


# --- 3) Nombre para UI ---
_SMALL = {"de", "del", "la", "el", "los", "las", "y", "o", "u", "con", "en", "al",
          "a", "un", "una", "para", "por", "sin", "the", "and"}


def title_es(name: str) -> str:
    text = fix_text_encoding(name)
    text = re.sub(r"^¿?c[oó]mo hacer\s+", "", text, flags=re.I).strip(" ¿?")
    letters = [c for c in text if c.isalpha()]
    if letters and (sum(c.islower() for c in letters) / len(letters) > 0.8):
        words = re.split(r"(\s+|-)", text)
        out = []
        first_word = True
        for w in words:
            if not w or re.fullmatch(r"\s+|-", w):
                out.append(w)
                continue
            low = w.lower()
            if (not first_word) and low in _SMALL:
                out.append(low)
            else:
                out.append(low[:1].upper() + low[1:] if low else w)
            first_word = False
        text = "".join(out)
    return text


# --- 4) Encoding residual sobre columnas de texto ya unificadas ---
for col in ["name", "ingredients", "instructions", "context", "recipe_text"]:
    if col in recipes.columns:
        recipes[col] = recipes[col].map(lambda x: x if pd.isna(x) else fix_text_encoding(x))

recipes["name"] = recipes["name"].map(title_es)
recipes["name_key"] = recipes["name"].map(normalize_key)
recipes["app_category"] = recipes.apply(infer_app_category, axis=1).replace("", np.nan)

time_fix = recipes.apply(clean_listed_time, axis=1)
recipes["prep_time_min"] = time_fix["prep_time_min"]
recipes["cook_time_min"] = time_fix["cook_time_min"]
recipes["listed_time_min"] = time_fix["listed_time_min"]
recipes["total_time_min"] = time_fix["total_time_min"]

# --- 5) Duplicado entre fuentes: misma receta, nos quedamos con la ficha más completa ---
def completeness_score(row) -> float:
    score = 0.0
    for col in ["instructions", "app_category", "app_difficulty", "total_time_min",
                "servings", "rating", "diet_tags"]:
        val = row[col]
        if pd.notna(val) and str(val).strip() not in ("", "False"):
            score += 1
    n_ing = row["n_ingredients_approx"]
    score += 0.05 * (n_ing if pd.notna(n_ing) else 0)
    return score


recipes["_score"] = recipes.apply(completeness_score, axis=1)
recipes = (
    recipes.sort_values(["_score", "rating", "n_steps_approx"], ascending=False)
    .drop_duplicates(["name_key", "ingredients_key"], keep="first")
    .drop(columns="_score")
    .copy()
)

recipes["recipe_text"] = recipes.apply(combine_for_recommender, axis=1)
recipes["n_ingredients_approx"] = recipes["ingredients"].map(count_structured_items)
recipes["n_steps_approx"] = recipes["instructions"].map(count_structured_items)
recipes = recipes.reset_index(drop=True)
recipes["recipe_id"] = np.arange(1, len(recipes) + 1)

print("Recetas antes de ajustes EDA:", n_before)
print("Recetas después (dedup cruzado):", len(recipes))
print("Categoría: cobertura", f"{100 * recipes['app_category'].notna().mean():.1f}%",
      "(antes", f"{100 * recipes_before['app_category'].notna().mean():.1f}%)")
print("Tiempo usable: cobertura",
      f"{100 * (recipes['total_time_min'].notna() & (recipes['total_time_min'] > 0)).mean():.1f}%")
print("Tiempos > 12 h en ficha:", int((recipes["total_time_min"] > 720).sum()))
print("Tiempos = 0 en ficha:", int((recipes["total_time_min"] == 0).sum()))
print("Mojibake restante en instrucciones:",
      int(recipes["instructions"].astype(str).str.contains("Ã", na=False).sum()))


Recetas antes de ajustes EDA: 28854
Recetas después (dedup cruzado): 28854
Categoría: cobertura 76.4% (antes 61.7%)
Tiempo usable: cobertura 76.2%
Tiempos > 12 h en ficha: 0
Tiempos = 0 en ficha: 0
Mojibake restante en instrucciones: 3


## 13. Limpieza de texto y normalización final

Hasta aquí el catálogo ya está unificado, pero el texto todavía trae ruido que el EDA no ve
y que contamina el recomendador (TF-IDF aprende "Tabasco" o "Galletas María" como señal):

| Paso | Qué corrige |
| --- | --- |
| 8a | Filas duplicadas por URL + nombre |
| 5 | Listas de Python que quedaron como texto (`['a', 'b']`) |
| 4 | Mojibake residual, emojis, entidades HTML, `<` `>` usados como separador |
| 1 | **Marcas comerciales** (incluye ®/™), electrodomésticos y utensilios de marca → término genérico |
| 2 | Referencias a Kiwilimón dentro de títulos y textos |
| 6 | Autopromoción: enlaces, redes sociales, "mira el vídeo", "sígueme" |
| 3 | `listed_time_min`: segundos → minutos, nulos/0 imputados desde `total_time_min` |
| 7 | Dificultades en minúsculas y `muy baja`/`muy alta` colapsadas |
| 8b | Recetas sin pasos: se reconstruyen desde `context`; si no hay de dónde, quedan nulas |
| 9 | **Instrucciones numeradas**: `1 paso \| 2 paso \| ...` |

Las marcas se reemplazan por un genérico en vez de borrarse (`Tabasco` → `salsa picante`,
`Coca Cola` → `refresco de cola`) para que la receta siga siendo cocinable.

Esta sección reemplaza al antiguo `limpiar_recetas.py`: todo corre desde `data/raw` en un solo notebook.


### 13.1 Utilidades y pasos 5 y 4 (listas, encoding, HTML)


In [15]:
import html
import json
from collections import Counter, OrderedDict

TEXT_COLS = ["name", "ingredients", "instructions"]
STATS = Counter()      # ocurrencias por regla  {(paso, etiqueta): n}
EXAMPLES = {}          # ejemplos antes/después por paso
AUDIT = Counter()      # candidatos a marca no cubiertos (preceden a ® / ™)
AUDIT_EX = {}


def tidy(t: str) -> str:
    """Limpieza de espacios/puntuación; solo se aplica a textos ya modificados."""
    segs = []
    for s in t.split(" | "):
        s = re.sub(r"[ \t\u00a0]{2,}", " ", s)
        s = re.sub(r"\(\s*[,;.:\-]?\s*\)", "", s)
        s = re.sub(r"\s+([,.;:!?])", r"\1", s)
        s = re.sub(r"([,;:])(\s*\1)+", r"\1", s)
        s = re.sub(r",\s*([.;:!?])", r"\1", s)
        s = re.sub(r"\b(?:de|del|con|para|y)\s*(?=[,.;:!?)]|$)", "", s, flags=re.I)
        s = re.sub(r"[ \t]{2,}", " ", s)
        s = re.sub(r",\s*$", "", s)
        s = s.strip(" ,;")
        if s:
            segs.append(s)
    return " | ".join(segs)


def _is_start(s: str, pos: int) -> bool:
    pre = s[:pos].rstrip()
    return (not pre) or pre[-1] in "|.!?:¡¿("


def rsub(rx, repl, t, step, label):
    """re.sub con conteo de ocurrencias y capitalización según contexto."""
    def f(m):
        r = repl(m) if callable(repl) else m.expand(repl)
        if r and _is_start(m.string, m.start()):
            r = r[0].upper() + r[1:]
        return r
    t2, n = rx.subn(f, t)
    if n:
        STATS[(step, label)] += n
    return t2


def renumber(t: str) -> str:
    segs = [s for s in t.split(" | ") if s.strip()]
    if segs and all(re.match(r"^\d+\s", s) for s in segs):
        segs = [f"{i} {re.sub(r'^\d+\s+', '', s)}" for i, s in enumerate(segs, 1)]
    return " | ".join(segs)


def snippet(old, new, w=55):
    p, m = 0, min(len(old), len(new))
    while p < m and old[p] == new[p]:
        p += 1
    q = 0
    while q < m - p and old[-1 - q] == new[-1 - q]:
        q += 1
    a = max(0, p - w)
    return old[a:len(old) - q + w][:230], new[a:len(new) - q + w][:230]


def apply_step(df, step, cols, func, tracker):
    """Aplica func(texto)->texto a las columnas; registra filas modificadas."""
    for col in cols:
        old = df[col]
        new = old.map(lambda t: func(t) if isinstance(t, str) else t)
        mask = old.notna() & new.ne(old)
        for idx in df.index[mask]:
            tracker[step][col].add(idx)
            ex = EXAMPLES.setdefault(step, [])
            if len(ex) < 3:
                a, b = snippet(old[idx], new[idx])
                ex.append((col, int(df.at[idx, "recipe_id"]), a, b))
        df[col] = new


In [16]:
def parse_pylist(t: str):
    s = t.strip()
    try:
        v = ast.literal_eval(s)
        if isinstance(v, (list, tuple)):
            return [str(x) for x in v]
    except Exception:
        pass
    inner = s.lstrip("[").rstrip("]").strip()           # lista truncada/malformada
    inner = inner.strip("'\" ")
    return re.split(r"""['"]\s*,\s*['"]""", inner)


def desparse_ingredients(t):
    if not re.match(r"^\s*\[", t):
        return t
    items = [re.sub(r"\s+", " ", i).strip() for i in parse_pylist(t)]
    STATS[("5", "ingredients: lista -> ' | '")] += 1
    return " | ".join(i for i in items if i)


def desparse_instructions(t):
    if not re.match(r"^\s*\[", t):
        return t
    items = [re.sub(r"\s+", " ", i).strip() for i in parse_pylist(t)]
    items = [re.sub(r"^\d+[.)]?\s+", "", i) for i in items if i]
    STATS[("5", "instructions: lista -> numerada")] += 1
    return " | ".join(f"{n} {i}" for n, i in enumerate(items, 1))


# ----------------------------------------------------------------------------
# PASO 4: mojibake, emojis, HTML, '<' '>'
# ----------------------------------------------------------------------------
_CONT = "".join(
    chr(b) if b in (0x81, 0x8D, 0x8F, 0x90, 0x9D) else bytes([b]).decode("cp1252")
    for b in range(0x80, 0xC0)
)
MOJI_RE = re.compile(
    rf"[ÂÃ][{_CONT}]|â[{_CONT}]{{2}}|ï[{_CONT}]{{2}}|ð[{_CONT}]{{3}}|[ÂÃâïð][{_CONT}]{{0,3}}"
)
EMOJI_RE = re.compile(
    "[\U0001F000-\U0001FAFF\u2600-\u27BF\u2B00-\u2BFF\u2300-\u23FF"
    "\uFE0E\uFE0F\u200D\u20E3\U000E0020-\U000E007F]+"
)
INVISIBLE_RE = re.compile("[\u200b-\u200f\u202a-\u202e\u2060\ufeff]")
ENT_MAP = {"&#8206;": "", "&#8207;": "", "&#8243;": '"', "&#8242;": "'",
           "&#133;": "...", "&hellip;": "...", "&nbsp;": " ", "&#160;": " "}
ENT_RE = re.compile(r"&(?:#\d+|#x[0-9a-fA-F]+|[A-Za-z][A-Za-z0-9]*);")
TAG_RE = re.compile(r"</?(?:br|p|b|i|u|em|strong|span|div|li|ul|ol|a|img)\b[^>]*>", re.I)


def _moji_fix(m):
    run = m.group(0)
    bs = bytearray()
    for ch in run:
        try:
            bs += ch.encode("cp1252")
        except UnicodeEncodeError:
            if ord(ch) < 256:
                bs.append(ord(ch))
            else:
                return ""
    try:
        return bytes(bs).decode("utf-8").replace("\xa0", " ")
    except UnicodeDecodeError:
        return ""         # secuencia irrecuperable (bytes perdidos) -> se elimina


def _math_to_plain(s):
    return "".join(unicodedata.normalize("NFKC", c) if 0x1D400 <= ord(c) <= 0x1D7FF else c for c in s)


def fix_encoding(t):
    o = t
    t2, n = MOJI_RE.subn(_moji_fix, t)
    if n:
        STATS[("4", "mojibake reparado/eliminado")] += n
    t = _math_to_plain(t2)
    t2, n = EMOJI_RE.subn("", t)
    if n:
        STATS[("4", "emojis/pictogramas eliminados")] += n
    t = t2
    for k, v in ENT_MAP.items():
        if k in t:
            STATS[("4", "entidad HTML " + k)] += t.count(k)
            t = t.replace(k, v)
    if ENT_RE.search(t):
        t, n = ENT_RE.subn(lambda m: html.unescape(m.group(0)), t)
        STATS[("4", "otras entidades HTML")] += n
    t, n = INVISIBLE_RE.subn("", t)
    if n:
        STATS[("4", "caracteres invisibles")] += n
    t, n = TAG_RE.subn(" ", t)
    if n:
        STATS[("4", "etiquetas HTML")] += n
    return tidy(t) if t != o else t


def fix_angle(t):
    if "<" not in t and ">" not in t:
        return t
    t = t.replace("<3", "\x00H\x00")           # corazón ASCII: se conserva
    t = re.sub(r"\bve<", "vez", t)             # errata detectada: 've<' = 'vez'

    def f(m):
        s, pre = m.string, (m.string[m.start() - 1] if m.start() else "")
        if m.start() == 0:
            return ""
        if m.end() == len(s):
            return "" if pre in ".!?" else "."
        return " " if pre in ".,;:!?" else ", "

    out = []
    for seg in t.split(" | "):
        seg2, n = re.subn(r"\s*[<>]+\s*", f, seg)
        STATS[("4", "'<' '>' usados como separador")] += n
        out.append(seg2)
    return tidy(" | ".join(out)).replace("\x00H\x00", "<3")


### 13.2 Marcas comerciales

1. Listas originales: marcas ambiguas (solo con ®/™), reemplazos genéricos y marcas que se eliminan siempre.
2. **Lista ampliada** (`add_brand`): bebidas, licores, chocolates, botanas, cereales, galletas, salsas y sazonadores
   detectados en el análisis de marcas (Coca-Cola, Baileys, Milo, Tabasco, Kit Kat, M&M's, Galletas María, Royal,
   Sriracha, Avecrem, Cointreau...). Cada marca declara su genérico y el género del artículo
   (`el Milo` → `la malta achocolatada`).
3. Se respetan usos que **no** son marca: `baño María`, `Royal Icing`, `Tabasco` como estado, `tarta de San Marcos`.


In [17]:
M = r"(?:\s?[®™])?"       # símbolo opcional tras la marca
PRE = r"(?:\b(?:de\s+la\s+marca|de\s+marca|marca|de|del|tipo)\s+)?"


def _alt(names):
    return "|".join(sorted(names, key=len, reverse=True))


# Marcas ambiguas (palabras comunes): solo se tocan si van seguidas de ® / ™
WITH_MARK = [
    r"D['’]?Gari", r"Gari", r"San\s+Juan", r"Tres\s+Estrellas", r"Del\s+Fuerte", r"Santa\s+Clara",
    r"Primavera", r"Extra\s+Special", r"Extra\s+Especial", r"Huichol", r"D[eo]l?\s+Campo", r"Domino",
    r"Berries\s+Paradise", r"San\s+Marcos", r"Gloria", r"Tortillinas", r"Luigi", r"Chantilly", r"Eva",
    r"Langa", r"Dolores", r"Ades", r"Evenflo(?:\s+Advance)?", r"Virgen\s+In[eé]s", r"Clavel", r"Iberia",
    r"Singles", r"Capullo", r"Ol[eé]ico", r"Plenia", r"Emergen-C", r"Silk", r"Zumba\s+Pica", r"Sos",
    r"Kinder", r"La\s+Moderna", r"Mark", r"Lybra", r"Bormioli", r"Saníssimo", r"Not\s+Butter",
    r"Cocina\s+Mestiza", r"Hummus", r"Caperucita", r"Valle\s+Frut", r"Fuerte\s+Origen",
]
WITH_MARK_REPL = [          # (patrón, reemplazo genérico) — requieren ® / ™
    (r"PAM|Pam", "aceite en aerosol"),
    (r"Sidral\s+Mundet", "refresco de manzana"),
    (r"Reddi\s+Wip", "crema batida en aerosol"),
    (r"(?:[Oo]lla\s+de\s+)?Presi[oó]n\s+Express", "olla de presión"),
]
# Marcas inequívocas: se eliminan siempre (lista del usuario + hallazgos del análisis)
ALWAYS_USER = [r"McCormick", r"Great\s+Value", r"Bimbo", r"Nestl[eéè]", r"Barilla", r"Heinz",
               r"Quaker", r"Danone", r"Alpina", r"Herdez"]
ALWAYS_FOOD = [r"D['’]Gari", r"Gamesa", r"Nutrioli", r"Oikos", r"Fage", r"Kikkoman", r"Yoplait",
               r"Danette", r"Skwinkles", r"Libanius", r"Jumex", r"Clemente\s+Jacques", r"La\s+Coste[ñn]a",
               r"Embasa", r"Valle\s+Frut", r"La\s+Asturiana", r"T[ií]a\s+Rosa", r"Do[ñn]a\s+Mar[ií]a",
               r"La\s+Botanera", r"La\s+Moderna", r"Best\s+Foods", r"Hellmann['’]?s", r"Kraft", r"Lala",
               r"Calahua", r"Proan", r"Del\s+Monte", r"Rexal"]
ALWAYS_APPLIANCE = [r"Oster", r"Braun", r"Vitamix", r"Weber", r"T-?fal", r"Tefal", r"KitchenAid",
                    r"Nutri\s?bullet", r"Instant\s+Pot", r"Ninja", r"Cuisinart", r"Magimix", r"Moulinex",
                    r"Cecotec", r"Crock-?\s?Pot", r"Royal\s+Prestige", r"Taurus", r"Philips",
                    r"Samsung(?:\s+Family\s*Hub)?"]

RX_WITH_MARK = re.compile(rf"{PRE}\b(?:{_alt(WITH_MARK)})\s?[®™]")
RX_WITH_MARK_REPL = [(re.compile(rf"\b(?:{p})\s?[®™]"), r) for p, r in WITH_MARK_REPL]
RX_ALWAYS_USER = re.compile(rf"{PRE}\b(?:{_alt(ALWAYS_USER)})\b{M}", re.I)
RX_ALWAYS_FOOD = re.compile(rf"{PRE}\b(?:{_alt(ALWAYS_FOOD)})\b{M}", re.I)
RX_ALWAYS_APP = re.compile(rf"{PRE}\b(?:{_alt(ALWAYS_APPLIANCE)})\b{M}", re.I)

AUTO_STRIP = True      # elimina el token capitalizado que precede a ®/™ si no está en las listas
RX_AUTO = re.compile(rf"{PRE}\b[A-ZÁÉÍÓÚÑ][^\s®™]*\s?[®™]")
RX_RESIDUAL = re.compile(r"((?:[A-ZÁÉÍÓÚÑ][^\s®™]*\s+){0,2}[A-ZÁÉÍÓÚÑ][^\s®™]*)\s?[®™]")

THERMO = r"Thermomix(?:\s*(?:TM\s?-?\s?\d{1,2}\b|TM\b|T\d{2}\b))?"


def _knorr(m):
    had_de = bool(re.match(r"(?:de|del)\s", m.group(0), re.I))
    tomate = bool(re.search(r"tomate", m.group(0), re.I))
    pre = re.split(r"[|.;]", m.string[max(0, m.start() - 45):m.start()])[-1]
    if re.search(r"\b(?:cubos?|cubitos?|pastillas?|sobres?|cuadritos?|cucharad(?:a|ita)s?|latas?|"
                 r"paquetes?|envases?|bolsas?)\s*$", pre, re.I):
        return "de caldo en polvo" + (" de tomate" if tomate else "")
    if re.search(r"caldo|consom|sazon|sopa|polvo|salsa|aderezo|crema|mayonesa|concentrado|pur[eé]|"
                 r"jugo|sabor", pre, re.I):
        return " de tomate" if tomate else ""
    return ("de " if had_de else "") + "caldo en polvo" + (" de tomate" if tomate else "")


def _royal(m):
    return m.group(1) if m.group(1) else "polvo para hornear"


REPLACEMENTS = [
    # (etiqueta, regex, reemplazo)
    ("Philadelphia -> queso crema",
     re.compile(rf"(?:queso|crema)(?:\s+(?:crema|de\s+queso))?\s+(?:tipo\s+)?(?:de\s+)?Philadelphia{M}", re.I), "queso crema"),
    ("Philadelphia -> queso crema",
     re.compile(rf"(?<!estilo )(?<!estilo de )(?<!ciudad de )(?:tipo\s+)?Philadelphia{M}", re.I), "queso crema"),
    ("Thermomix -> robot de cocina",
     re.compile(rf"\bde\s+(?:la\s+)?{THERMO}{M}", re.I), "del robot de cocina"),
    ("Thermomix -> robot de cocina",
     re.compile(rf"\bla\s+{THERMO}{M}", re.I), "el robot de cocina"),
    ("Thermomix -> robot de cocina",
     re.compile(rf"\buna\s+{THERMO}{M}", re.I), "un robot de cocina"),
    ("Thermomix -> robot de cocina",
     re.compile(rf"\b(en|con|para|desde|sin|al|usando)\s+{THERMO}{M}", re.I), r"\1 el robot de cocina"),
    ("Thermomix -> robot de cocina",
     re.compile(rf"{THERMO}{M}", re.I), "robot de cocina"),
    ("Mycook/Bimby/Cookeo -> robot de cocina",
     re.compile(rf"\b(?:Mycook(?:\s+(?:Touch|Easy|Pro))?(?:\s+de\s+Taurus)?|Bimby|Cookeo|Thermochef){M}", re.I), "robot de cocina"),
    ("Maizena/Maicena -> fécula de maíz (dup.)",
     re.compile(rf"(f[eé]cula\s+de\s+ma[ií]z)\s+(?:marca\s+)?(?:Maizena|Maicena){M}", re.I), r"\1"),
    ("Maizena/Maicena -> fécula de maíz",
     re.compile(rf"\b(?:Maizena|Maicena){M}", re.I), "fécula de maíz"),
    ("Nutella -> crema de avellanas y cacao",
     re.compile(rf"(?:crema\s+de\s+avellanas?(?:\s+y\s+cacao)?\s+|chocolate\s+para\s+untar\s+)?(?:tipo\s+)?Nutella{M}", re.I),
     "crema de avellanas y cacao"),
    ("Oreo -> galletas de chocolate rellenas",
     re.compile(rf"\bgalleta\s+(?:tipo\s+)?Oreo{M}", re.I), "galleta de chocolate rellena"),
    ("Oreo -> galletas de chocolate rellenas",
     re.compile(rf"\b(?:1|una?)\s+Oreo{M}\b", re.I), "1 galleta de chocolate rellena"),
    ("Oreo -> galletas de chocolate rellenas",
     re.compile(rf"(?:galletas?|galletitas?)\s+(?:tipo\s+)?Oreos?{M}", re.I), "galletas de chocolate rellenas"),
    ("Oreo -> galletas de chocolate rellenas",
     re.compile(rf"\bOreos?{M}", re.I), "galletas de chocolate rellenas"),
    ("Knorr/Maggi/Avecrem -> caldo en polvo",
     re.compile(rf"(?:\b(?:de|del|marca|de\s+la\s+marca|de\s+marca)\s+)?\b(?:Knorr|Maggis?|Maggui|Avecrem)\b{M}"
                r"(?:\s+(?:Suiza|Tomate))?", re.I), _knorr),
    ("Royal (polvo para hornear)",
     re.compile(r"\bRoyal\b(?=\s+(?:polvo\s+(?:para|de)\s+hornear|para\s+hornear|levadura))\s*"), ""),
    ("Royal (polvo para hornear)",
     re.compile(r"((?:polvo\s+(?:para|de)\s+hornear|levadura\s+qu[ií]mica)\s+)?(?<!Kir )\bRoyal\b"
                r"(?!\s+(?i:Prestige|Icing|Canin|Oak|Jelly|Gala))" + M), _royal),
    ("La Lechera -> leche condensada",
     re.compile(rf"(leche\s+condensada\s+)?\bLa\s+Lechera{M}", re.I),
     lambda m: "leche condensada" if not m.group(1) else m.group(1)),
    ("Splenda -> edulcorante",
     re.compile(rf"(edulcorante|endulzante)\s+(?:bajo\s+en\s+calor[ií]as\s+)?(?:de\s+)?E?Splenda{M}", re.I), r"\1"),
    ("Splenda -> edulcorante",
     re.compile(rf"\bE?splenda{M}", re.I), "edulcorante"),
    ("Samsung Smart Oven -> horno",
     re.compile(r"Smart\s+Oven(?:\s+HotBlast)?(?:\s+de\s+Samsung)?", re.I), "horno"),
    ("Maggie (salsa/jugo) -> eliminada",
     re.compile(r"(?<=salsa )Maggie|(?<=jugo )Maggie|(?<=sazonador )Maggie|(?<=caldo )Maggie", re.I), ""),
    ("demaicena -> de fécula de maíz",
     re.compile(r"\bde(?:maizena|maicena)\b", re.I), "de fécula de maíz"),
    ("Pyrex -> refractario (dup.)",
     re.compile(rf"\b(refractario|recipiente|molde|fuente|charola|vasija)\s+(?:de\s+)?Pyrex{M}", re.I), r"\1"),
    ("Silpat -> tapete de silicón (dup.)",
     re.compile(rf"\b(?:tapete|alfombrilla|l[aá]mina|hoja)\s+(?:de\s+)?Silpat{M}", re.I), "tapete de silicón"),
    ("Lékué -> recipiente de silicón (dup.)",
     re.compile(rf"\b(?:recipiente|estuche|molde)\s+(?:de\s+)?L[eé]ku[eé]{M}", re.I), "recipiente de silicón"),
    ("Pyrex -> refractario",
     re.compile(rf"\bPyrex{M}", re.I), "refractario"),
    ("Silpat -> tapete de silicón",
     re.compile(rf"\bSilpat{M}", re.I), "tapete de silicón"),
    ("Lékué -> recipiente de silicón",
     re.compile(rf"\bL[eé]ku[eé]{M}", re.I), "recipiente de silicón"),
    ("Teflon -> antiadherente",
     re.compile(rf"\b(?:de\s+)?Tefl[oó]n{M}", re.I), "antiadherente"),
    ("Sidral Mundet -> refresco de manzana",
     re.compile(rf"\bSidral(?:\s+Mundet)?{M}", re.I), "refresco de manzana"),
]
ALWAYS_RULES = [("marca secundaria eliminada (lista del usuario)", RX_ALWAYS_USER),
                ("marca de alimento eliminada (hallazgo)", RX_ALWAYS_FOOD),
                ("marca de electrodoméstico/utensilio eliminada", RX_ALWAYS_APP)]

# En títulos las marcas de caldo se quitan (no se reemplazan por "caldo en polvo")
RX_KNORR_NAME = re.compile(rf"{PRE}\b(?:Knorr|Maggis?|Maggui|Avecrem)\b{M}(?:\s+(?:Suiza|Tomate))?", re.I)


In [18]:
B_ = r"(?<![\wáéíóúñÁÉÍÓÚÑ])"      # inicio de palabra (también con acentos)
E_ = r"(?-i:(?![a-záéíóúñ0-9_])(?!(?<=[A-ZÁÉÍÓÚÑ])[A-ZÁÉÍÓÚÑ]))"   # fin de palabra; admite "srirachaMayonesa" pero no "FANTASIA"
Q_ = r"['’´`]?"                     # apóstrofo opcional (Baileys, M&M's, Hershey's)

_ART_F = {"el": "la", "del": "de la", "al": "a la", "un": "una", "unos": "unas", "los": "las"}
_ART_M = {"la": "el", "de la": "del", "a la": "al", "una": "un", "unas": "unos", "las": "los"}
_SMALL_T = {"de", "del", "la", "el", "los", "las", "y", "o", "con", "en", "al", "a", "para", "sin", "un", "una"}

EXTRA = []        # (etiqueta, regex, reemplazo, alcance)   alcance: "all" | "text" (no títulos)
_TRIG = []        # patrones que activan el bloque; evita correr ~150 regex sobre textos sin marcas
_GENERICS = set()


def _article_rule(label, brand, generic, g, scope):
    table = _ART_F if g == "f" else _ART_M
    arts = "|".join(sorted((re.escape(a).replace(r"\ ", r"\s+") for a in table), key=len, reverse=True))
    rx = re.compile(rf"\b({arts})\s+(?:{brand}){E_}{M}", re.I)
    return (label, rx, lambda m: table[" ".join(m.group(1).lower().split())] + " " + generic, scope)


def add_brand(label, brand, generic, g="m", cat=None, scope="all", plain=True):
    """Marca -> término genérico.

    `cat`: palabra(s) de categoría que suelen preceder a la marca ("licor Cointreau");
    se reemplaza toda la frase por el genérico para no duplicar ("licor licor de naranja").
    `g`: género del genérico, para corregir el artículo ("el Milo" -> "la malta achocolatada").
    """
    _TRIG.append(brand)
    if generic and " " in generic:
        _GENERICS.add(generic)
    if cat:
        EXTRA.append((label, re.compile(
            rf"\b(?:{cat})\s+(?:(?:de|del|marca|tipo|estilo)\s+)?(?:{brand}){E_}{M}", re.I), generic, scope))
    if not plain:
        return
    if generic:
        EXTRA.append(_article_rule(label, brand, generic, g, scope))
        EXTRA.append((label, re.compile(rf"{B_}(?:{brand}){E_}{M}", re.I), generic, scope))
    else:
        EXTRA.append((label, re.compile(rf"{PRE}{B_}(?:{brand}){E_}{M}", re.I), "", scope))


def add_special(label, pattern, repl, scope="all", flags=re.I, trig=None):
    _TRIG.append(trig or pattern)
    EXTRA.append((label, re.compile(pattern, flags), repl, scope))


COLA = rf"(?:Coca[\s-]?Cola|Pepsi(?:[\s-]?Cola)?)(?:\s+(?:Zero|Light|Diet|Life))?"
SODA_CAT = r"refrescos?(?:\s+de\s+cola)?|gaseosas?(?:\s+de\s+cola)?|bebidas?(?:\s+gaseosas?)?(?:\s+de\s+cola)?|sodas?(?:\s+de\s+cola)?|colas?"

# ---- Refrescos --------------------------------------------------------------------
add_special("Coca-Cola/Pepsi + 'a la/de la' -> refresco de cola",
            rf"\b(a|de|con|en)\s+la\s+{COLA}{E_}{M}",
            lambda m: {"a": "al", "de": "del"}.get(m.group(1).lower(), m.group(1).lower() + " el") + " refresco de cola",
            trig=COLA)
add_special("Coca-Colas (plural) -> refrescos de cola", rf"{B_}(?:Coca[\s-]?Cola|Pepsi)s{E_}", "refrescos de cola", trig=r"(?:Coca[\s-]?Cola|Pepsi)s")
add_brand("Coca-Cola/Pepsi -> refresco de cola", COLA, "refresco de cola", "m", cat=SODA_CAT)
add_special("'lata de Coca' -> refresco de cola",
            rf"\b((?:lata|botella|vaso|taza|litro|mililitros?|ml)s?\s+(?:de\s+)?)(?-i:Coca)(?!\s*-?\s*Cola){E_}",
            r"\1refresco de cola", trig=r"(?-i:Coca)\b")
add_brand("Sprite/7Up -> refresco de limón", r"Sprite|7[\s-]?Up|Seven[\s-]?Up", "refresco de limón", "m", cat=SODA_CAT)
add_brand("Fanta -> refresco de naranja", r"Fanta", "refresco de naranja", "m", cat=SODA_CAT)
add_brand("Squirt -> refresco de toronja", r"Squirt", "refresco de toronja", "m", cat=SODA_CAT)
add_brand("Red Bull/Monster -> bebida energética", r"Red\s?Bull|Monster(?:\s+Energy)?", "bebida energética", "f",
          cat=r"bebidas?\s+energ[eé]ticas?")
add_brand("Lipton -> té", r"Lipton", "té", "m", cat=r"t[eé](?:\s+negro|\s+helado)?")
add_brand("Nestea -> té helado en polvo", r"Nestea", "té helado en polvo", "m", cat=r"t[eé](?:\s+helado)?")

# ---- Licores ---------------------------------------------------------------------
add_special("'tipo Nescafé' -> eliminado", rf"\s*\((?:tipo|como)\s+Nescaf[eé]\)", "", trig=r"Nescaf")
add_brand("Nescafé -> café instantáneo", r"Nescaf[eé]", "café instantáneo", "m", cat=r"caf[eé](?:\s+(?:soluble|instant[aá]neo))?")
add_brand("Baileys -> licor de crema irlandesa", rf"Bailey{Q_}s(?:\s+Irish\s+Cream)?(?:\s+Original)?",
          "licor de crema irlandesa", "m", cat=r"licor(?:\s+de\s+crema(?:\s+irlandesa)?)?|crema(?:\s+irlandesa)?")
add_brand("Cointreau/Grand Marnier -> licor de naranja", r"Cointreau|Cointrau|Grand\s+Marnier",
          "licor de naranja", "m", cat=r"licor(?:\s+de\s+naranja)?")
add_brand("Kahlúa/Tía María -> licor de café", r"Kahl[uú]a|T[ií]a\s+Mar[ií]a", "licor de café", "m",
          cat=r"licor(?:\s+de\s+caf[eé])?")
add_brand("Frangelico -> licor de avellana", r"Frangelico", "licor de avellana", "m",
          cat=r"licor(?:\s+de\s+avellanas?)?")
add_brand("Midori -> licor de melón", r"Midori", "licor de melón", "m", cat=r"licor(?:\s+de\s+mel[oó]n)?")
add_brand("Malibú -> licor de coco", r"Malib[uú]", "licor de coco", "m", cat=r"licor(?:\s+de\s+coco)?|ron(?:\s+de\s+coco)?")
add_brand("Chambord -> licor de frambuesa", r"Chambord", "licor de frambuesa", "m", cat=r"licor(?:\s+de\s+frambuesas?)?")
add_brand("Pernod -> licor de anís", r"Pernod", "licor de anís", "m", cat=r"licor(?:\s+de\s+an[ií]s)?")
add_brand("Licor 43 -> licor de vainilla", r"Licor\s+43|Cuarenta\s+y\s+Tres", "licor de vainilla", "m")
add_brand("Campari -> aperitivo amargo", r"Campari", "aperitivo amargo", "m", cat=r"aperitivo(?:\s+amargo)?")
add_brand("Bacardí/Havana Club -> ron", r"Bacard[ií](?:\s+(?:Blanco|Carta\s+Blanca|Limón))?|Havana\s+Club", "ron", "m", cat=r"ron")
add_brand("Absolut -> vodka", r"Absolut", "vodka", "m", cat=r"vodka")
add_brand("Tanqueray/Bombay -> ginebra", r"Tanqueray|Bombay(?:\s+Sapphire)?", "ginebra", "f", cat=r"ginebra")
add_brand("Jack Daniel's/Johnnie Walker -> whisky", rf"Jack\s+Daniel{Q_}s|Johnnie\s+Walker", "whisky", "m", cat=r"whisk(?:e)?y")
add_special("Martini (vermut) -> vermut",
            rf"\b(?:vermut|vermouth)\s+Martini(?:\s+(?:Rosso|Bianco|Dry|Extra\s+Dry|Rojo|Blanco))?{E_}"
            rf"|{B_}Martini\s+(?:Rosso|Bianco|Extra\s+Dry|Rojo){E_}", "vermut", scope="text",
            trig=r"Martini")

# ---- Chocolates, dulces y botanas ----------------------------------------------------
add_special("Kinder Sorpresa -> huevo de chocolate sorpresa",
            rf"\bhuevos?\s+Kinder(?:\s+sorpresa)?{E_}|{B_}Kinder\s+Sorpresa{E_}", "huevo de chocolate sorpresa", trig=r"Kinder")
add_special("Kinder Bueno -> chocolate con avellanas",
            rf"{B_}Kinder\s+Bueno(?:\s+(?:White|Blanco|Mini))?{E_}", "chocolate con avellanas", trig=r"Kinder")
add_brand("Kinder -> chocolate", r"Kinder(?:\s+(?:Delice|Maxi|Chocolate|Country|Joy|Happy\s+Hippo))?", "chocolate", "m")
add_brand("Kit Kat -> chocolate con barquillo", r"Kit[\s-]?Kats?", "chocolate con barquillo", "m", cat=r"chocolates?")
add_brand("M&M's -> grageas de chocolate de colores", rf"M\s?&\s?M(?:{Q_}s)?", "grageas de chocolate de colores", "f",
          cat=r"chocolates?|grageas?|confites?|dulces?|lunetas?")
add_brand("Snickers -> barra de chocolate con cacahuate", r"Snickers", "barra de chocolate con cacahuate", "f")
add_brand("Twix -> barra de chocolate con galleta", r"Twix", "barra de chocolate con galleta", "f")
add_brand("Toblerone -> chocolate con turrón", r"Toblerone", "chocolate con turrón", "m", cat=r"chocolate")
add_brand("Ferrero Rocher -> bombones de chocolate y avellana", r"Ferrero(?:\s+Rocher)?|Rocher", "bombones de chocolate y avellana", "m",
          cat=r"bombones?|chocolates?")
add_brand("Reese's -> chocolate con crema de cacahuate", rf"Reese(?:{Q_}s)?", "chocolate con crema de cacahuate", "m",
          cat=r"chocolates?|barras?|copas?|(?:crema|mantequilla)\s+de\s+cacahuate")
add_brand("Lindt/Kellogg's -> (eliminada)", rf"Lindt|Kellogg{Q_}s", "", "m")
add_brand("Gansito -> pastelito relleno", r"Gansitos?", "pastelito relleno", "m", cat=r"pastelitos?|pastelillos?")
add_brand("Abuelita/Ibarra/Carlos V -> chocolate de mesa", r"Abuelita|Ibarra|Carlos\s+V", "chocolate de mesa", "m",
          cat=r"chocolates?(?:\s+de\s+mesa)?|tablillas?(?:\s+de\s+chocolate)?")
add_brand("Nesquik -> chocolate en polvo", r"Nesqu[ií]c?k", "chocolate en polvo", "m",
          cat=r"chocolate(?:\s+en\s+polvo)?|cacao(?:\s+en\s+polvo)?|cocoa|polvo\s+(?:para\s+preparar\s+)?(?:de\s+)?chocolate|"
              r"saborizante(?:\s+de\s+chocolate)?|bebida\s+(?:de\s+)?chocolate")
add_special("Nesquik (jarabe) -> jarabe de chocolate",
            rf"\b(?:jarabe|sirope|salsa)(?:\s+de\s+chocolate)?\s+(?:de\s+)?Nesqu[ií]c?k{E_}{M}", "jarabe de chocolate", trig=r"Nesqu")
add_brand("Hershey's Kisses -> chocolates pequeños", rf"Hershey{Q_}s?\s+(?:Kisses|Besos)|Kisses", "chocolates pequeños", "m",
          cat=r"chocolates?|besos")
add_special("Hershey's (jarabe) -> jarabe de chocolate",
            rf"\b(?:jarabe|sirope|salsa)(?:\s+de\s+chocolate)?\s+(?:de\s+)?Hershey{Q_}s?{E_}{M}", "jarabe de chocolate", trig=r"Hershey")
add_special("Hershey's -> chocolate/cocoa genérico",
            rf"\b((?:chocolate|cocoa|cacao)(?:\s+en\s+polvo)?)\s+(?:de\s+|marca\s+)?Hershey{Q_}s?{E_}{M}", r"\1", trig=r"Hershey")
add_brand("Hershey's -> chocolate", rf"Hershey{Q_}s?", "chocolate", "m")
add_brand("Nocilla -> crema de avellanas y cacao", r"Nocilla", "crema de avellanas y cacao", "f",
          cat=r"crema(?:\s+de\s+(?:avellanas?|cacao))*(?:\s+y\s+cacao)?|chocolate\s+para\s+untar")
add_brand("Cola Cao -> cacao en polvo", r"Cola[\s-]?Cao", "cacao en polvo", "m", cat=r"cacao(?:\s+en\s+polvo)?")
add_brand("Doritos -> totopos de maíz", r"Doritos", "totopos de maíz", "m", cat=r"totopos|frituras|botanas?")
add_brand("Cheetos -> frituras de maíz con queso", r"Cheetos", "frituras de maíz con queso", "f", cat=r"frituras|botanas?")
add_brand("Takis -> frituras de maíz enrolladas", r"Takis", "frituras de maíz enrolladas", "f", cat=r"frituras|botanas?")
add_brand("Sabritas/Pringles/Ruffles -> papas fritas", r"Sabritas|Pringles|Ruffles", "papas fritas", "f",
          cat=r"papas(?:\s+fritas)?|botanas?")
add_brand("Tajín -> chile en polvo con limón", r"Taj[ií]n", "chile en polvo con limón", "m",
          cat=r"chile(?:\s+en\s+polvo)?(?:\s+con\s+lim[oó]n)?|sazonador")

# ---- Cereales, galletas y panes ------------------------------------------------------
add_brand("Cheerios -> cereal de avena en aros", r"Cheerios", "cereal de avena en aros", "m", cat=r"cereal(?:\s+de\s+avena)?")
add_brand("Corn Flakes -> hojuelas de maíz", r"Corn[\s-]?Flakes", "hojuelas de maíz", "f", cat=r"cereal(?:\s+de)?|hojuelas|copos")
add_brand("Choco Krispis -> cereal de arroz inflado con chocolate", r"Choco[\s-]?Krispis", "cereal de arroz inflado con chocolate", "m",
          cat=r"cereal")
add_brand("Maseca -> harina de maíz nixtamalizado", r"Maseca", "harina de maíz nixtamalizado", "f",
          cat=r"harina(?:\s+de\s+ma[ií]z)?(?:\s+nixtamalizad[oa])?|masa\s+harina")
add_brand("Mazola -> aceite de maíz", r"Mazola", "aceite de maíz", "m", cat=r"aceite(?:\s+de\s+ma[ií]z)?")
add_brand("Knox -> gelatina sin sabor", r"Knox", "gelatina sin sabor", "f", cat=r"gelatina(?:\s+sin\s+sabor)?|grenetina")
add_brand("Carnation -> leche evaporada", r"Carnation", "leche evaporada", "f", cat=r"leche(?:\s+evaporada)?")
add_brand("Lyncott -> queso crema", r"Lyncott", "queso crema", "m", cat=r"queso(?:\s+crema)?")
add_brand("Alpura -> leche", r"Alpura", "leche", "f", cat=r"leche")
add_brand("Gran Capitán -> queso", r"Gran\s+Capit[aá]n", "queso", "m", cat=r"queso", plain=False)

for _b, _gp, _gs in [
    (r"Ritz(?:\s+(?:Crackers|Bits))?|Ducales|Saladitas|Wheat\s+Thins", "galletas saladas", "galleta salada"),
    (r"Chips\s+Ahoy", "galletas con chispas de chocolate", "galleta con chispas de chocolate"),
    (r"Emperador", "galletas rellenas", "galleta rellena"),
    (r"(?:Lotus\s+)?Biscoff", "galletas de caramelo especiado", "galleta de caramelo especiado"),
]:
    add_special(f"{_gp} (marca eliminada, singular)", rf"\bgalleta\s+(?:(?:tipo|de|marca)\s+)?(?:{_b}){E_}{M}", _gs, trig=_b)
    add_brand(f"-> {_gp}", _b, _gp, "f", cat=r"galletas?", plain=_b != r"Emperador")

add_brand("Thins (pan) -> pan delgado", r"(?-i:Thins)", "pan delgado", "m", cat=r"pan")

# ---- Galletas María (tipo de galleta, no se tocan 'baño María' ni nombres propios) ------
add_special("Galletas María -> galletas dulces",
            rf"\bgalletas\s+(?:(?:tipo|de\s+la\s+marca|de|marca)\s+)?Mar[ií]as?{E_}", "galletas dulces", trig=r"galletas?\s+(?:\w+\s+)*Mar[ií]a")
add_special("Galleta María -> galleta dulce",
            rf"\bgalleta\s+(?:(?:tipo|de\s+la\s+marca|de|marca)\s+)?Mar[ií]as?{E_}", "galleta dulce", trig=r"galleta\s+Mar[ií]a")
add_special("'Digestive/Chiquilín/galletas dulces o María' -> sin María",
            rf"\b(Digestive|Chiquil[ií]n|galletas?\s+(?:dulces?|saladas?))\s*(?:o|y|/)\s*Mar[ií]as?{E_}", r"\1", trig=r"Mar[ií]as?")
add_special("'galletas ..., tipo María' -> galletas ...",
            rf"(\bgalletas?\b[^|()]{{0,30}}?)[,\s]+(?:tipo|estilo)\s+Mar[ií]as?{E_}", r"\1", trig=r"(?:tipo|estilo)\s+Mar[ií]a")
add_special("'tipo María' -> tipo galleta dulce",
            rf"\b(tipo|estilo)\s+Mar[ií]as?{E_}", r"\1 galleta dulce", trig=r"(?:tipo|estilo)\s+Mar[ií]a")
add_special("Marías (plural) -> galletas dulces",
            rf"(?<!Tres )(?<!Ave )(?<!Santa )\bMar[ií]as{E_}", "galletas dulces", trig=r"Mar[ií]as")

# ---- Salsas y sazonadores ------------------------------------------------------------
add_special("'salsa tipo Tabasco' -> salsa picante",
            rf"\bsalsa\s+(?:picante\s+)?(?:tipo|estilo|como)\s+Tabasco{E_}{M}", "salsa picante", trig=r"Tabasco")
add_special("'ají/chile Tabasco' -> ají/chile picante",
            rf"\b(aj[ií]|chile)\s+Tabasco{E_}{M}", r"\1 picante", scope="text", trig=r"Tabasco")
add_special("'salsa (de) Tabasco' -> salsa picante",
            rf"\bsalsa(?:\s+picante)?(?:\s+de)?\s+Tabasco{E_}{M}", "salsa picante", trig=r"Tabasco")
add_special("', tipo Tabasco' -> eliminado",
            rf",?\s*\b(?:tipo|estilo|como)\s+Tabasco{E_}{M}", "", scope="text", trig=r"Tabasco")
add_special("'el/del Tabasco' -> la/de la salsa picante",
            rf"\b(el|del)\s+Tabasco{E_}{M}", lambda m: {"el": "la", "del": "de la"}[m.group(1).lower()] + " salsa picante",
            scope="text", trig=r"Tabasco")
add_special("Tabasco -> salsa picante (solo texto; en títulos 'con Tabasco')",
            rf"(?<!estado de )(?<!regi[oó]n de )(?<!ciudad de )(?<!chocolate de )(?<!cacao de ){B_}Tabascos?{E_}{M}",
            "salsa picante", scope="text", trig=r"Tabasco")
add_special("Tabasco -> salsa picante (títulos)", rf"(?<=\bcon ){B_}Tabasco{E_}{M}", "salsa picante", scope="name", trig=r"Tabasco")
add_brand("Sriracha -> salsa picante", r"Sriracha", "salsa picante", "f", cat=r"salsa(?:\s+picante)?(?:\s+de\s+chile)?")
add_brand("Cholula/Tapatío/Frank's RedHot -> salsa picante", rf"Cholula|Tapat[ií]o|Frank{Q_}s\s+Red\s?Hot|Red\s?Hot", "salsa picante", "f",
          cat=r"salsa(?:\s+picante)?")
add_special("Valentina -> salsa picante",
            rf"\bsalsa(?:\s+picante)?\s+(?:de\s+)?Valentina{E_}"
            rf"|(?<=Catsup y )Valentina{E_}|(?<=Ketchup o )Valentina{E_}|(?<=como )valentina{E_}"
            rf"|(?<=salsa roja, )valentina{E_}|(?<=picante, )valentina{E_}", "salsa picante", trig=r"valentina")
add_special("maggie (minúscula, salsa sazonadora) -> salsa sazonadora",
            rf"{B_}(?-i:maggie){E_}", "salsa sazonadora", flags=re.I, trig=r"maggie")
add_special("'Mis Recetas Al Horno' (sazonador) -> eliminado",
            r"\bMis\s+Recetas\s+(?=Al\s+(?:Horno|Tomate))", "", scope="text", trig=r"Mis\s+Recetas")
add_special("Orlando (tomate frito) -> tomate frito",
            rf"\b(tomate\s+frito|salsa\s+de\s+tomate|tomates?\s+fritos?)\s+Orlando(?:\s+(?:Casero|Cl[aá]sic[oa]))?{E_}", r"\1",
            scope="text", trig=r"Orlando")
add_special("Orlando (tomate) -> tomate frito",
            rf"\b(tomates?)\s+Orlando(?:\s+(?:Casero|Cl[aá]sic[oa]))?{E_}", "tomate frito", scope="text", trig=r"Orlando")

# ---- Milo, La Lechera y Royal (los casos que dejaba la lista original) -------------------
add_special("Galletas Milo -> galletas de malta achocolatada",
            rf"\bgalletas?\s+(?-i:Milo){E_}", "galletas de malta achocolatada", trig=r"(?-i:Milo)")
add_brand("Milo -> malta achocolatada", r"(?-i:Milo|MILO)", "malta achocolatada", "f",
          cat=r"malta(?:\s+achocolatada)?|chocolate\s+(?:en\s+polvo|instant[aá]neo)|cacao\s+en\s+polvo|cocoa|"
              r"polvo\s+de\s+chocolate|bebida\s+(?:de\s+)?chocolate")
add_special("milo (minúscula, en medida) -> malta achocolatada",
            r"\b((?:tazas?|cucharad(?:a|ita)s?|gramos?)\s+(?:soperas?\s+)?de\s+(?:chocolate,\s+)?)milo\b", r"\1malta achocolatada",
            trig=r"\bmilo\b")
add_special("el milo -> la malta achocolatada",
            r"\b(?:y\s+)?\bel\s+milo\b", lambda m: m.group(0)[:-len("el milo")] + "la malta achocolatada", trig=r"\bel\s+milo\b")
add_special("alimento en polvo milo -> malta achocolatada",
            r"\balimento\s+en\s+polvo\s+milo\b", "malta achocolatada", trig=r"\bmilo\b")
add_special("chocolate, milo -> chocolate, malta achocolatada",
            r"\bchocolate,\s+milo\b", "chocolate, malta achocolatada", trig=r"\bmilo\b")
add_special("Lechera -> leche condensada",
            rf"\b(de|con|y|o|la|una)\s+Lechera{E_}{M}", r"\1 leche condensada", trig=r"Lechera")
add_special("Lechera (suelta) -> leche condensada",
            rf"(?<!olla )(?<!jarra )(?<!c[aá]ntaro )(?<!bid[oó]n ){B_}Lechera{E_}{M}", "leche condensada", trig=r"Lechera")
add_special("Royal + gelatina/flan -> gelatina/flan",
            rf"\b(gelatina|flan|natilla|pud[ií]n|bud[ií]n|postre)\s+(?:marca\s+|de\s+)?Royal{E_}(?!\s+icing)", r"\1", trig=r"Royal")
add_special("Royal + polvo/levadura -> polvo/levadura",
            rf"\b(polvo\s+(?:para|de)\s+hornear|levadura(?:\s+qu[ií]mica)?)\s*[,(]?\s*(?:tipo\s+|marca\s+|de\s+)?Royale?{E_}(?!\s+icing)\)?",
            r"\1", trig=r"Royale?")
add_special("'la Royal' -> el polvo para hornear",
            rf"\bla\s+Royal{E_}(?!\s+(?:icing|jelly|gala|canin|oak|prestige))", "el polvo para hornear", trig=r"Royal")
add_special("royal (minúscula, sin 'icing') -> polvo para hornear",
            rf"(?<!kir ){B_}royal{E_}(?!\s+(?:icing|jelly|gala|canin|oak|prestige|wedding|blue|flush|family))", "polvo para hornear",
            trig=r"royal")

TRIGGER_RX = re.compile("|".join(f"(?:{t})" for t in _TRIG), re.I)
DUP_RX = re.compile(
    r"\b(" + "|".join(re.escape(g) for g in sorted(_GENERICS, key=len, reverse=True)) + r")(?:\s*(?:,|/|\bo\b|\by\b|\bu\b)\s*\1)+\b", re.I)


def _title_phrase(s, at_start):
    out = []
    for i, w in enumerate(s.split(" ")):
        out.append(w if (w in _SMALL_T and (i or not at_start) ) else w[:1].upper() + w[1:])
    return " ".join(out)


def _shape(rep, is_name):
    """Adapta el reemplazo al formato del texto original (MAYÚSCULAS, Título, minúscula)."""
    def f(m):
        r = rep(m) if callable(rep) else m.expand(rep)
        if not r:
            return r
        txt = m.group(0)
        letters = [c for c in txt if c.isalpha()]
        if len(letters) > 3 and all(c.isupper() for c in letters):
            return r.upper()
        words = [w for w in txt.split() if w[:1].isalpha()]
        if is_name or (len(words) > 1 and all(w[0].isupper() for w in words)):
            return _title_phrase(r, m.start() == 0)
        return r
    return f


def clean_brands_extra(t, is_name=False):
    if not TRIGGER_RX.search(t):
        return t
    for label, rx, rep, scope in EXTRA:
        if (scope == "text" and is_name) or (scope == "name" and not is_name):
            continue
        t = rsub(rx, _shape(rep, is_name), t, "1", label)
    return DUP_RX.sub(lambda m: m.group(1), t)


In [19]:
def clean_brands(t, is_name=False):
    o = t
    if is_name:     # "Thermomix Paella" -> "Paella" ; "Paella (Thermomix)" -> "Paella"
        t = rsub(re.compile(rf"^\s*{THERMO}{M}\s*[-–:]?\s+", re.I), "", t, "1", "Thermomix al inicio del título removido")
        t = rsub(re.compile(rf"\s*[-–:(]?\s*(?<!con )(?<!en )(?<!para )(?<!de ){THERMO}{M}\)?\s*$", re.I), "", t, "1", "Thermomix al final del título removido")
        t = rsub(RX_KNORR_NAME, "", t, "1", "Knorr/Maggi/Avecrem removido del título")
    # 1a) marcas ambiguas, solo si llevan ® / ™
    for rx, rep in RX_WITH_MARK_REPL:
        t = rsub(rx, rep, t, "1", "marca con ®/™ -> genérico")
    t = rsub(RX_WITH_MARK, "", t, "1", "marca ambigua + ®/™ eliminada")
    # 1b) reemplazos genéricos
    for label, rx, rep in REPLACEMENTS:
        t = rsub(rx, _shape(rep, is_name), t, "1", label)
    # 1b-bis) lista ampliada de marcas (bebidas, licores, chocolates, galletas, salsas...)
    t = clean_brands_extra(t, is_name)
    # 1c) eliminación de marcas
    for label, rx in ALWAYS_RULES:
        t = rsub(rx, "", t, "1", label)
    # 1d) auditoría de marcas no cubiertas (aún con ®/™)
    for m in RX_RESIDUAL.finditer(t):
        k = m.group(1).strip()
        AUDIT[k] += 1
        AUDIT_EX.setdefault(k, t[max(0, m.start() - 40):m.end() + 30])
    # 1e) marcas no listadas: token capitalizado inmediatamente antes de ®/™
    if AUTO_STRIP:
        t = rsub(RX_AUTO, "", t, "1", "marca no listada (token previo a ®/™) eliminada automáticamente")
    # 1f) símbolos ® ™ y TM suelto
    t, n = re.subn(r"\s?[®™]", "", t)
    if n:
        STATS[("1", "símbolos ®/™ eliminados")] += n
    t, n = re.subn(r"(?<=[a-záéíóúñ0-9])TM\b|\(?\bTM\s?-?\s?\d{1,2}\b\)?", "", t)
    if n:
        STATS[("1", "'TM' / modelo TM5 eliminado")] += n
    t = re.sub(r"\b(queso crema)(?:\s+(?:de\s+)?queso crema)+", r"\1", t, flags=re.I)
    t = re.sub(r"\b(recipiente de silicón)(?:\s+(?:de\s+)?\1)+", r"\1", t, flags=re.I)
    t = re.sub(r"\b(edulcorante|endulzante)\s+(?:o|y|/)\s+\1\b", r"\1", t, flags=re.I)
    return tidy(t) if t != o else t


### 13.3 Kiwilimón y autopromoción


In [20]:
KIWI = r"kiwi\s?lim[oó]n"
K_RULES = [
    ("URL de kiwilimon", re.compile(r"\(?\s*(?:https?://|www\.)\S*kiwilimon\S*\s*\)?", re.I)),
    ("paréntesis con 'kiwilimon'", re.compile(rf"\s*\([^()|]*{KIWI}[^()|]*\)", re.I)),
    ("'ver receta ... en kiwilimon'",
     re.compile(rf",?\s*\bver\s+(?:la\s+)?receta[^|.;()]{{0,80}}?(?:en|de|del)\s+{KIWI}", re.I)),
    ("'receta de ... en kiwilimon'",
     re.compile(rf"\s*(?:\b(?:con|seg[uú]n|de)\s+)?(?:la\s+|esta\s+)?\breceta\s+(?:de\s+|del\s+|que\s+est[aá]\s+)?"
                rf"[^|.;()]{{0,80}}?\s*(?:en|de|del)\s+{KIWI}", re.I)),
    ("'los clásicos de kiwilimón'",
     re.compile(rf"[¡!]?\s*[^|.!¡?]*cl[aá]sicos\s+de\s+{KIWI}\s*[!.]?", re.I)),
]
K_FALLBACK = re.compile(rf"[^|.!?]*{KIWI}[^|.!?]*[.!?]?\s*", re.I)


def _prev_word(s, pos):
    w = re.findall(r"(\w+)\s*$", s[:pos])
    return w[0].lower() if w else ""


def _kiwi_name(m):
    if m.group(1):                      # "Kiwilimón" en una sola palabra
        return ""
    if _prev_word(m.string, m.start()) in {"de", "con", "y", "e", "o", "al", "el", "la", "un", "una", "del", "jugo", "sabor"}:
        return m.group(0)               # "Mojito de Kiwi Limón" = fruta kiwi + limón
    return ""


def clean_kiwi_name(t):
    o = t
    rx = re.compile(r"\s*[-–—|:]?\s*(?:\b(Kiwilim[oó]n)\b|\bKiwi\s+Lim[oó]n\b)", re.I)
    t = rsub(rx, _kiwi_name, t, "2", "'Kiwilimón' removido del título")
    return tidy(t) if t != o else t


def clean_kiwi_text(t, numbered=False):
    o = t
    if not re.search(KIWI, t, re.I):
        return t
    for label, rx in K_RULES:
        t = rsub(rx, "", t, "2", label)
    if re.search(KIWI, t, re.I):                       # mención residual -> se quita la oración
        segs = []
        for seg in t.split(" | "):
            seg, n = K_FALLBACK.subn("", seg)
            if n:
                STATS[("2", "oración residual con 'kiwilimon' eliminada")] += n
            segs.append(seg)
        t = " | ".join(segs)
    t = tidy(t)
    return renumber(t) if (numbered and " | " in o and t.count(" | ") != o.count(" | ")) else t


# ----------------------------------------------------------------------------
# PASO 6: autopromoción y enlaces
# ----------------------------------------------------------------------------
URL_RX = re.compile(r"(?:(?:Imagen|Foto|Truco|V[ií]deo|Video|Fuente)\s*:\s*)?"
                    r"(?:https?://\S+|\bwww\.\S+|\b[\w-]+\.(?:com|net|org|es|mx|blogspot\.com|wordpress\.com)\b(?:/\S*)?)", re.I)
CTA_RX = re.compile(r"consulta|visita|m[aá]s recetas|mira|descubre|encuentra|sigue|v[ií]deo|video|canal|blog|"
                    r"pincha|haz clic|click|entra", re.I)
PROMO_RX = re.compile(
    r"\b(?:mi|mis|nuestro|nuestra|nuestros|nuestras)\s+(?:blogs?|canal|perfil|p[aá]gina|web|sitio|youtube|instagram|facebook|tiktok|redes|recetario)\b"
    r"|\bs[ií]gu(?:enos|eme|enme|anos)\b|\bsuscr[ií]b(?:e|ete|ir|irte|ase|anse)\b|\bcampanita\b"
    r"|\b(?:instagram|facebook|tiktok|pinterest|telegram|whatsapp|twitter|youtube)\b"
    r"|\bvisit(?:a|ar|es|e|ando)\s+(?:\w+\s+){0,2}?(?:blog|canal|web|p[aá]gina|perfil|sitio)\b"
    r"|\bvisit(?:a|ar|es|e|ando)\s+(?-i:[A-ZÁÉÍÓÚÑ])"
    r"|[\w.+-]+@[\w-]+\.\w+|(?<![\w])@\w{3,}"
    r"|\bp[aá]sate\s+por\b|\becha\s+un\s+vistazo\s+a\s+mi\b"
    r"|\b(?:mira|descubre|ve|ver)\s+(?:aqu[ií]\s+)?el\s+v[ií]deo\b|\bv[ií]deo\s+aqu[ií]\b"
    r"|¿te\s+interesan\s+m[aá]s\s+recetas|\bsi\s+te\s+ha\s+gustado\s+esta\s+receta\b", re.I)
PROMO_PAREN = re.compile(r"\s*\([^()|]*(?:\b(?:mi|nuestro)\s+(?:perfil|blogs?|canal|recetario)|instagram|youtube|facebook|tiktok)[^()|]*\)", re.I)
SENT_SPLIT = re.compile(r"(?<=[.!?])\s*(?=[A-ZÁÉÍÓÚÑ¡¿])|(?<=[.!?])\s+")
CLAUSE_SPLIT = re.compile(r"(?<=[,;])\s+")


def clean_promo(t, numbered=False):
    o = t
    if not (URL_RX.search(t) or PROMO_RX.search(t)):
        return t
    t, n = PROMO_PAREN.subn("", t)
    if n:
        STATS[("6", "paréntesis promocional eliminado (ej. '(receta en mi perfil)')")] += n
    segs, dropped = [], 0
    for seg in t.split(" | "):
        keep = []
        if not numbered:        # ingredientes: se quita solo la cláusula promocional, no el ingrediente
            parts = CLAUSE_SPLIT.split(seg)
            if len(parts) > 1 and not PROMO_RX.search(parts[0]) and not URL_RX.search(parts[0]):
                good = [p_ for p_ in parts if not PROMO_RX.search(p_) and not (URL_RX.search(p_) and CTA_RX.search(p_))]
                if len(good) < len(parts):
                    dropped += 1
                    STATS[("6", "cláusula promocional removida de un ingrediente")] += 1
                    seg = " ".join(good)
        for sent in SENT_SPLIT.split(seg):
            if URL_RX.search(sent):
                if CTA_RX.search(sent) or PROMO_RX.search(sent):
                    dropped += 1
                    STATS[("6", "oración con enlace + llamada a la acción")] += 1
                    continue
                sent = URL_RX.sub("", sent)
                STATS[("6", "URL suelta eliminada")] += 1
            elif PROMO_RX.search(sent):
                dropped += 1
                STATS[("6", "oración autopromocional eliminada")] += 1
                continue
            keep.append(sent)
        seg2 = " ".join(keep).strip()
        if re.fullmatch(r"\d*\W*", seg2):
            seg2 = ""
        if seg2:
            segs.append(seg2)
    t = tidy(" | ".join(segs))
    return renumber(t) if (numbered and dropped and " | " in o) else t


### 13.4 Numeración de instrucciones y columnas derivadas

Formato `1 paso | 2 paso | ...` (el mismo que ya parsea `split_steps` en el backend, así la app no duplica números).
Un solo párrafo sin separadores se divide en oraciones; una numeración previa consecutiva se reinicia para
que todas las recetas queden con 1..N sin huecos.


In [21]:
def make_key(s):
    s = unicodedata.normalize("NFKD", str(s).lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", s)).strip()


def build_recipe_text(r):
    cols = ["name", "ingredients", "category_tags", "diet_tags", "meal_type", "app_category", "country", "difficulty"]
    return " | ".join(str(r[c]) for c in cols if pd.notna(r[c]) and str(r[c]).strip())


_ABBR = {"gr", "grs", "kg", "min", "mins", "seg", "ml", "lt", "lts", "cda", "cdas", "cdta", "cdtas", "cdita", "cditas",
         "aprox", "ej", "etc", "sra", "sr", "dr", "dra", "vs", "oz", "lb", "cc", "máx", "mín", "núm", "no", "ud", "uds"}
_SENT_BOUND = re.compile(r"(?<=[.!?])\s+(?=[A-ZÁÉÍÓÚÑ¡¿])")
_LEAD_NUM = re.compile(r"^(\d+)[.)]?\s+")


def split_sentences(text):
    """Divide un párrafo en oraciones sin cortar abreviaturas ('20 min. Retira') ni 'Paso 1. Mezcla'."""
    pieces, start = [], 0
    for m in _SENT_BOUND.finditer(text):
        tokens = text[start:m.start()].split()
        w = tokens[-1].lower().lstrip("([¡¿\"'") if tokens else ""
        if re.fullmatch(r"\d+[.)]", w) or (w.endswith(".") and w[:-1] in _ABBR) or re.fullmatch(r"[a-záéíóúñ]\.", w):
            continue
        pieces.append(text[start:m.start()].strip())
        start = m.end()
    pieces.append(text[start:].strip())
    merged = []
    for p in pieces:
        if merged and len(p) < 12:      # fragmentos mínimos ("¡Buen provecho!") se pegan al paso anterior
            merged[-1] += " " + p
        else:
            merged.append(p)
    if len(merged) > 1 and len(merged[0]) < 12:
        merged = [merged[0] + " " + merged[1]] + merged[2:]
    return [p for p in merged if p]


def number_steps(t):
    """'paso | paso' -> '1 paso | 2 paso'.

    - La numeración previa solo se descarta si arranca en 1 y es consecutiva (2+ pasos);
      así no se recorta un paso que empieza con '2 tazas de...'.
    - Un paso suelto de puro número/puntuación ('| 5') se elimina.
    - Si la receta es un solo párrafo, se parte en oraciones y cada una es un paso.
    Mismo formato que ya entiende `src/text_utils.split_steps` (N + espacio).
    """
    segs = [s.strip() for s in t.split(" | ") if s.strip()]
    segs = [s for s in segs if not re.fullmatch(r"\d*\W*", s)]
    if not segs:
        return t
    run = 0
    for i, s in enumerate(segs, 1):
        m = _LEAD_NUM.match(s)
        if m and int(m.group(1)) == i:
            run = i
        else:
            break
    if run >= 2:
        segs[:run] = [_LEAD_NUM.sub("", s, count=1) for s in segs[:run]]
    if len(segs) == 1:
        segs = split_sentences(segs[0]) or segs
    segs = [s for s in segs if s]
    return " | ".join(f"{i} {s}" for i, s in enumerate(segs, 1))


### 13.5 Ejecución sobre el catálogo


In [22]:
AUTO_STRIP = True            # quita el token capitalizado previo a ®/™ aunque la marca no esté en las listas
STRICT_URL_DEDUP = False     # True: elimina TODAS las filas con URL repetida (incluye recetas distintas)

STEPS = ["8a", "5", "4", "1", "2", "6", "3", "7", "8b", "9"]
STATS.clear(); EXAMPLES.clear(); AUDIT.clear(); AUDIT_EX.clear()

df = recipes.copy()
orig = df.copy()
n_in, cols_in = len(df), list(df.columns)
tracker = {k: {c: set() for c in df.columns} for k in STEPS}
notes = {}

# ---- 8a. Deduplicación por URL ------------------------------------------------------
dup_url = df["url"].duplicated(keep="first")
dup_url_name = df.duplicated(subset=["url", "name_key"], keep="first")
notes["8a"] = {
    "filas con URL repetida (regla estricta)": int(dup_url.sum()),
    "filas duplicadas reales (misma URL y mismo nombre)": int(dup_url_name.sum()),
    "modo aplicado": "ESTRICTO (solo URL)" if STRICT_URL_DEDUP else "SEGURO (URL + nombre)",
}
drop_mask = dup_url if STRICT_URL_DEDUP else dup_url_name
removed_8a = int(drop_mask.sum())
df = df[~drop_mask].copy()

# ---- 5. Listas de Python -> texto -------------------------------------------------
apply_step(df, "5", ["ingredients"], desparse_ingredients, tracker)
apply_step(df, "5", ["instructions"], desparse_instructions, tracker)

# ---- 4. Mojibake / HTML / emojis / separadores -----------------------------------------
apply_step(df, "4", TEXT_COLS, fix_encoding, tracker)
apply_step(df, "4", ["ingredients", "instructions"], fix_angle, tracker)

# ---- 1. Marcas comerciales --------------------------------------------------------------
apply_step(df, "1", ["name"], lambda t: clean_brands(t, True), tracker)
apply_step(df, "1", ["ingredients", "instructions"], clean_brands, tracker)

# ---- 2. Kiwilimón ----------------------------------------------------------------------
apply_step(df, "2", ["name"], clean_kiwi_name, tracker)
apply_step(df, "2", ["ingredients"], clean_kiwi_text, tracker)
apply_step(df, "2", ["instructions"], lambda t: clean_kiwi_text(t, True), tracker)
apply_step(df, "2", ["meal_type", "category_tags"],
           lambda t: tidy(re.sub(rf"\s*\b{KIWI}\b", "", t, flags=re.I)) if re.search(KIWI, t, re.I) else t, tracker)

# ---- 6. Autopromoción y enlaces ----------------------------------------------------------
ctx_promo = int(df["context"].fillna("").map(lambda t: bool(PROMO_RX.search(t))).sum())
apply_step(df, "6", ["ingredients"], clean_promo, tracker)
apply_step(df, "6", ["instructions"], lambda t: clean_promo(t, True), tracker)
notes["6"] = {"filas con promo en 'context' (columna que se elimina al final)": ctx_promo}

# ---- 3. Tiempos ---------------------------------------------------------------------------
lt, tt = df["listed_time_min"].copy(), df["total_time_min"]
secs = lt.notna() & tt.notna() & np.isclose(lt, tt * 60)
df.loc[secs, "listed_time_min"] = lt[secs] / 60
tracker["3"]["listed_time_min"].update(df.index[secs])
need = df["listed_time_min"].isna() | (df["listed_time_min"] == 0)
can = need & tt.notna() & (tt > 0)
df.loc[can, "listed_time_min"] = tt[can]
tracker["3"]["listed_time_min"].update(df.index[can])
zero_left = df["listed_time_min"] == 0
df.loc[zero_left, "listed_time_min"] = np.nan
tracker["3"]["listed_time_min"].update(df.index[zero_left])
notes["3"] = {
    "segundos -> minutos (listed = total*60)": int(secs.sum()),
    "nulos/0 imputados desde total_time_min": int(can.sum()),
    "0.0 sin total disponible -> NaN": int(zero_left.sum()),
    "nulos que NO se pueden imputar (total_time_min también nulo)": int(df["listed_time_min"].isna().sum()),
}

# ---- 7. Dificultades -----------------------------------------------------------------------
d_new = df["difficulty"].str.strip().str.lower().replace({"muy baja": "baja", "muy alta": "alta"})
a_new = d_new.map({"baja": "facil", "media": "intermedio", "alta": "reto"}).fillna(df["app_difficulty"])
ch_d = df["difficulty"].notna() & d_new.ne(df["difficulty"])
ch_a = df["app_difficulty"].notna() & a_new.ne(df["app_difficulty"])
notes["7"] = {
    "difficulty llevada a minúsculas": int((df["difficulty"].notna() & (df["difficulty"] != df["difficulty"].str.lower())).sum()),
    "difficulty 'muy baja'/'muy alta' colapsada": int(df["difficulty"].isin(["muy baja", "muy alta"]).sum()),
    "app_difficulty corregida": int(ch_a.sum()),
}
df["difficulty"], df["app_difficulty"] = d_new, a_new
tracker["7"]["difficulty"].update(df.index[ch_d])
tracker["7"]["app_difficulty"].update(df.index[ch_a])

# ---- 8b. Sin instrucciones: se reconstruyen desde `context`; si no se puede, quedan nulas ---------
blank = df["instructions"].isna() | df["instructions"].fillna("").str.strip().eq("") | ~df["has_instructions"].astype(bool)
rebuilt = 0
for idx in df.index[blank]:
    c = df.at[idx, "context"]
    steps = []
    if isinstance(c, str) and len(re.findall(r"(?:^|\s)\d+[.)]\s", c)) >= 3:
        steps = [s.strip() for s in re.split(r"(?:^|\s)\d+[.)]\s", c) if s.strip()]
    if steps:
        df.at[idx, "instructions"] = " | ".join(steps)
        tracker["8b"]["instructions"].add(idx)
        rebuilt += 1
    else:
        df.at[idx, "instructions"] = np.nan
notes["8b"] = {"filas sin instrucciones": int(blank.sum()), "reconstruidas desde context": rebuilt,
               "se quedan nulas (el backend las devuelve como lista vacía)": int(blank.sum()) - rebuilt}

# ---- 9. Numeración de instrucciones ---------------------------------------------------------------
apply_step(df, "9", ["instructions"], number_steps, tracker)

# ---- Columnas derivadas ----------------------------------------------------------------------------
df = df.drop(columns=["context"])
df["name_key"] = df["name"].map(make_key)
df["ingredients_key"] = df["ingredients"].map(make_key)

empty_ing = df["ingredients"].fillna("").str.strip().eq("")
empty_name = df["name"].fillna("").str.strip().eq("")
dup_after = df.duplicated(["name_key", "ingredients_key"], keep="first")
drop_final = empty_ing | empty_name | dup_after
notes["final"] = {
    "sin ingredientes/nombre tras la limpieza": int((empty_ing | empty_name).sum()),
    "duplicados nuevos (misma receta una vez quitadas las marcas)": int((dup_after & ~(empty_ing | empty_name)).sum()),
}
df = df[~drop_final].copy()

df["has_ingredients"] = True
df["has_instructions"] = df["instructions"].notna()
df["recipe_text"] = df.apply(build_recipe_text, axis=1)
changed_any = pd.Index(sorted(set().union(*[tracker[s]["ingredients"] for s in tracker])))
changed_any = changed_any.intersection(df.index[orig.loc[df.index, "n_ingredients_approx"].notna()])
df.loc[changed_any, "n_ingredients_approx"] = df.loc[changed_any, "ingredients"].map(lambda t: float(t.count(" | ") + 1))
df["n_steps_approx"] = df["instructions"].map(lambda t: float(t.count(" | ") + 1) if isinstance(t, str) else np.nan)

df = df.reset_index(drop=True)
df["recipe_id"] = np.arange(1, len(df) + 1)
out_cols = [c for c in cols_in if c != "context"]
recipes = df[out_cols].copy()

print(f"Filas: {n_in:,} -> {len(recipes):,} | columnas: {len(cols_in)} -> {len(out_cols)} (sin `context`)")


Filas: 28,854 -> 28,853 | columnas: 30 -> 29 (sin `context`)


### 13.6 Qué cambió


In [23]:
STEP_TITLES = {
    "8a": "Deduplicación por URL", "5": "Listas de Python -> texto",
    "4": "Mojibake, emojis, HTML y '<' '>'", "1": "Marcas, electrodomésticos y ®/™",
    "2": "Referencias a Kiwilimón", "6": "Autopromoción y enlaces", "3": "Tiempos (listed_time_min)",
    "7": "Dificultades", "8b": "Recetas sin instrucciones", "9": "Numeración de instrucciones",
}
rows = []
for k, title in STEP_TITLES.items():
    per_col = {c: len(v) for c, v in tracker[k].items() if v}
    n_rows = removed_8a if k == "8a" else len(set().union(*tracker[k].values()))
    rows.append({"paso": k, "qué hace": title, "registros modificados": n_rows,
                 "por columna": ", ".join(f"{c}: {n:,}" for c, n in per_col.items()) or "—"})
display(pd.DataFrame(rows).set_index("paso"))

print("Notas por paso:")
for k, v in {**notes}.items():
    print(f"  [{k}]", "; ".join(f"{a}: {b}" for a, b in v.items()))

print("\nOcurrencias por regla de marcas (top 25):")
top_rules = pd.Series({lbl: n for (s, lbl), n in STATS.items() if s == "1"}).sort_values(ascending=False).head(25)
display(top_rules.to_frame("ocurrencias"))

print("Marcas detectadas por ®/™ que no están en las listas (se eliminaron automáticamente; top 15):")
display(pd.DataFrame(AUDIT.most_common(15), columns=["marca_detectada", "ocurrencias"]))


,qué hace,registros modificados,por columna
paso,,,
8a,Deduplicación por URL,1,—
5,Listas de Python -> texto,26,"ingredients: 2, instructions: 24"
4,"Mojibake, emojis, HTML y '<' '>'",313,"name: 5, ingredients: 46, instructions: 292"
1,"Marcas, electrodomésticos y ®/™",5429,"name: 856, ingredients: 4,231, instructions: 4,169"
2,Referencias a Kiwilimón,16,"name: 8, ingredients: 2, instructions: 6, meal_type: 8, category_tags: 8"
6,Autopromoción y enlaces,2179,"ingredients: 12, instructions: 2,174"
3,Tiempos (listed_time_min),155,listed_time_min: 155
7,Dificultades,9430,"difficulty: 9,430"
8b,Recetas sin instrucciones,0,—


Notas por paso:
  [8a] filas con URL repetida (regla estricta): 827; filas duplicadas reales (misma URL y mismo nombre): 1; modo aplicado: SEGURO (URL + nombre)
  [6] filas con promo en 'context' (columna que se elimina al final): 292
  [3] segundos -> minutos (listed = total*60): 51; nulos/0 imputados desde total_time_min: 0; 0.0 sin total disponible -> NaN: 104; nulos que NO se pueden imputar (total_time_min también nulo): 6686
  [7] difficulty llevada a minúsculas: 9019; difficulty 'muy baja'/'muy alta' colapsada: 411; app_difficulty corregida: 0
  [8b] filas sin instrucciones: 4; reconstruidas desde context: 0; se quedan nulas (el backend las devuelve como lista vacía): 4
  [final] sin ingredientes/nombre tras la limpieza: 0; duplicados nuevos (misma receta una vez quitadas las marcas): 0

Ocurrencias por regla de marcas (top 25):


,ocurrencias
marca ambigua + ®/™ eliminada,1836
Maizena/Maicena -> fécula de maíz,1654
Thermomix -> robot de cocina,1551
marca no listada (token previo a ®/™) eliminada automáticamente,1404
marca de alimento eliminada (hallazgo),1253
marca secundaria eliminada (lista del usuario),828
Knorr/Maggi/Avecrem -> caldo en polvo,714
Oreo -> galletas de chocolate rellenas,562
Galletas María -> galletas dulces,417
Philadelphia -> queso crema,403


Marcas detectadas por ®/™ que no están en las listas (se eliminaron automáticamente; top 15):


,marca_detectada,ocurrencias
0,Campo,41
1,Mermelada Smucker’s,31
2,Zwan,28
3,Fud,27
4,FUD,27
5,Hunt´S,26
6,Goya,24
7,Conejos,18
8,Lucas,15
9,Danonino,14


### 13.7 Marcas antes y después

Conteo de **recetas** que mencionan cada marca (una receta cuenta una vez por columna).
`Tabasco` puede quedar en pocos títulos donde es el estado mexicano, y `Royal` solo como `Royal Icing`.


In [24]:
BRAND_CHECK = {
    "Coca-Cola / Pepsi": r"coca[\s-]?cola|pepsi",
    "Baileys": r"bailey['’´`]?s",
    "Milo": r"(?-i:\bMilo\b)",
    "La Lechera": r"\bla\s+lechera\b|\b(?:de|con|y|o)\s+lechera\b",
    "Tabasco": r"\btabasco\b",
    "Kit Kat": r"kit[\s-]?kat",
    "Kinder": r"\bkinder\b",
    "Galletas María": r"galletas?\s+(?:tipo\s+|de\s+)?mar[ií]as?\b|(?<!tres )\bmar[ií]as\b",
    "M&M": r"\bm\s?&\s?m\b",
    "Nesquik": r"nesqu[ií]c?k",
    "Maggi": r"\bmaggi\b|\bmaggui\b|\bmaggie\b(?!\s+wakeham)",
    "Royal (polvo para hornear)": r"(?<!kir )\broyal\b(?!\s+(?:icing|jelly|gala|canin|oak|prestige))",
    "Pepsi": r"pepsi",
    "Cheerios": r"cheerios",
    "Cointreau": r"cointreau",
    "Sriracha": r"sriracha",
    "Avecrem": r"avecrem",
    "Grand Marnier": r"grand\s+marnier",
    "Nescafé": r"nescaf[eé]",
    "Galletas Ritz": r"\britz\b",
    "Leche Alpura": r"alpura",
    "Corn Flakes": r"corn[\s-]?flakes",
    "Salsa Orlando": r"tomate\s+(?:frito\s+)?orlando|salsa\s+orlando",
    "Oreo": r"\boreo",
    "Knorr": r"knorr",
    "Nutella": r"nutella",
    "Philadelphia": r"philadelphia",
    "Maizena": r"maiz?e?na\b",
    "Splenda": r"splenda",
}
BRAND_RX = {b: re.compile(p, re.I) for b, p in BRAND_CHECK.items()}


def brand_report(frame):
    out = {}
    for b, rx in BRAND_RX.items():
        row = {}
        for col in ["name", "ingredients", "instructions"]:
            s = frame[col].fillna("")
            row[col] = int(s.str.contains(rx).sum())
        out[b] = row
    return pd.DataFrame(out).T


before = brand_report(orig)
after = brand_report(recipes)
brand_summary = before.join(after, lsuffix=" (antes)", rsuffix=" (después)")
display(brand_summary)
print("Menciones residuales (sin contar Tabasco, que sigue en 3 títulos donde es el estado mexicano):",
      int(after.drop(index="Tabasco").to_numpy().sum()))


,name (antes),ingredients (antes),instructions (antes),name (después),ingredients (después),instructions (después)
Coca-Cola / Pepsi,15,40,33,0,0,0
Baileys,7,14,15,0,0,0
Milo,6,6,7,0,0,0
La Lechera,6,70,65,0,0,0
Tabasco,5,165,140,3,0,1
Kit Kat,4,5,6,0,0,0
Kinder,8,10,12,0,0,0
Galletas María,28,285,157,0,0,0
M&M,5,12,11,0,0,0
Nesquik,3,10,10,0,0,0


Menciones residuales (sin contar Tabasco, que sigue en 3 títulos donde es el estado mexicano): 0


## 14. Validación final y archivo processed

Se mantiene la regla de negocio: toda receta tiene nombre e ingredientes. El tiempo de
ficha nunca es 0 ni mayor a 12 horas. Las categorías, si existen, entran en el contrato
de la app.

También se comprueba que las instrucciones queden numeradas `1..N`, que no queden marcas de la lista de seguimiento y que el resultado sea **un único CSV** en `data/processed/`.


In [25]:
assert recipes["recipe_id"].is_unique
assert recipes["name"].ne("").all()
assert recipes["ingredients"].ne("").all()
assert not recipes.duplicated(["name_key", "ingredients_key"]).any()
assert recipes["total_time_min"].dropna().gt(0).all()
assert recipes["total_time_min"].dropna().le(12 * 60).all()

valid_cat = {"desayuno", "comida", "cena", "postre", "bebida"}
valid_diff = {"facil", "intermedio", "reto"}
assert set(recipes["app_category"].dropna().unique()).issubset(valid_cat)
assert set(recipes["app_difficulty"].dropna().unique()).issubset(valid_diff)

assert "context" not in recipes.columns


def _numbered(text):
    return all(re.match(rf"^{i}\s", step) for i, step in enumerate(text.split(" | "), 1))


assert recipes["instructions"].dropna().map(_numbered).all(), "hay instrucciones sin numerar"
assert not recipes["instructions"].eq("Revisión requerida").any()
leaks = after.drop(index="Tabasco")
assert leaks.to_numpy().sum() == 0, leaks[leaks.sum(axis=1) > 0]

print("Catálogo final:", recipes.shape)
print("\nPor fuente:")
display(recipes["source"].value_counts().to_frame("recetas"))
print("\nCategoría app:")
display(recipes["app_category"].value_counts(dropna=False).to_frame("recetas"))
print("\nNulos en campos de ficha:")
display(
    recipes[["app_category", "app_difficulty", "total_time_min", "servings", "instructions"]]
    .isna().sum().to_frame("nulos")
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
recipes.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("\nGuardado:", OUTPUT_PATH)
print("Tamaño MB:", round(OUTPUT_PATH.stat().st_size / 1024 / 1024, 2))

print("\nMuestra de instrucciones numeradas:")
for _, row in recipes[recipes["instructions"].notna()].sample(2, random_state=7).iterrows():
    print(f"\n[{row['recipe_id']}] {row['name']}\n{row['instructions'][:420]}...")

print("\nArchivos en data/processed:", sorted(p.name for p in PROCESSED_DIR.iterdir() if not p.name.startswith(".")))


Catálogo final: (28853, 29)

Por fuente:


,recetas
source,
RecetasGratis,18186
Kiwilimon,9019
Yanuq,791
Cookpad,720
Directo al Paladar,92
El Mueble,37
Vitónica,8



Categoría app:


,recetas
app_category,
comida,14247
NaN,6813
postre,4989
desayuno,1582
bebida,917
cena,305



Nulos en campos de ficha:


,nulos
app_category,6813
app_difficulty,6052
total_time_min,6878
servings,15604
instructions,4



Guardado: /Users/i./Documents/DIPLOMADO/MODULO_5/chefWise/data/processed/recetas_unificadas_limpias.csv
Tamaño MB: 59.04

Muestra de instrucciones numeradas:

[27679] Aguacate Antillano
1 Poner en la licuadora hielo y todos los ingredientes excepto el aguacate. | 2 Licuar hasta que el hielo se triture bien. | 3 Sacar la mitad de la mezcla y reservarla. | 4 Licuar el aguacate con la otra mitad. | 5 En una copa servir la mitad de la mezcla del aguacate. | 6 Encima cuidando que no se mezclen, la otra mitad que se reservó. | 7 Servir....

[13425] Galletas de Fantasmas
1 Precalentar el horno a 180ºC. | 2 En un recipiente y con una batidora acreme la mantequilla y el azúcar. ya que estén mezclados, agregar el huevo y el extracto de vainilla. | 3 Mezcle bien el harina y el polvo para hornear. | 4 Agregue harina a la mezcla de mantequilla. Revolviendo muy bien después de cada adición. La última parte de harina, mezclela a mano; debe de estar la masa dura. | 5 Divida la masa en dos bol...

Arc

## 15. Qué queda para el modelo y la app

- **Contenido:** `recipe_text` (nombre + ingredientes + tags). No incluye pasos a propósito.
- **Texto limpio:** sin marcas comerciales (genéricos en su lugar), sin promoción de blogs/redes y con los pasos numerados.
  Lo que el catálogo no cubre se puede ampliar con `add_brand(...)` en la sección 13.2.
- **Filtros de producto:** `app_category`, `app_difficulty`, `total_time_min` (tiempo usable),
  `servings`, `country`, `diet_tags`.
- **Mapeo a Angular:** `recipe_id` → `id`, `app_category` → `category`, `total_time_min` →
  `minutes`, `app_difficulty` → `difficulty`; partir ingredientes y pasos por ` | ` (los pasos ya vienen como `1 texto`; `split_steps` quita el número).
- **Aún incompleto a propósito:** porciones en Kiwilimon, categoría en una parte de RecetasGratis
  sin pistas en el nombre, y `cena` (casi nadie etiqueta cena vs comida). Eso se puede completar
  después con un clasificador, no con reglas frágiles.
- **Interacciones:** favoritos, cocinado y “con lo que tengo” siguen siendo la señal que falta
  para un híbrido; el frontend ya deja el lugar para registrarlas.

Siguiente paso: `02_eda_e_ingenieria_de_variables.ipynb` (EDA y variables) y `03_recomendador_content_based.ipynb` (modelo).
